![image.png](https://i.imgur.com/a3uAqnb.png)

# MLOps Lab: Deploy, Monitor and Retire a Model

This lab is the practical half of **Lecture 02 - MLOps Foundations**. The deck closes with two exercises on its *Where to Practise This* slide: **tracking and tuning** with MLflow and Optuna, and **serving and drift**. Here they are fused into one loop, run end to end:

**train → track → tune → package → register → deploy → call → monitor → promote → roll back → destroy**

The model is deliberately boring: a scikit-learn pipeline that reads handwritten digits and trains in seconds on a CPU. The point is everything around it, which is the deck's opening argument ("*the ML code is the small box*", after Sculley et al., [Hidden Technical Debt in Machine Learning Systems, NeurIPS 2015](https://proceedings.neurips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html)).

### What you will have built by the end

- A **reproducible training run**: a pinned data snapshot, a seed, an environment fingerprint, all logged to **MLflow**.
- An **Optuna** study with pruning, every trial logged, and your own evidence on whether pruning pays.
- A **packaged model**: the whole pipeline in one `skops` file, plus its preprocessing code, a schema and a model card.
- A **model registry** on the Hugging Face Hub: immutable versions (commits), readable tags, and a mutable `champion` alias.
- A **live web app with a public URL**: a drawing pad you can open on your phone, and an HTTP API that this notebook calls over the wire.
- A **prediction log**, a **drift monitor** (PSI and KS) and an alert that fires only on a *sustained* shift.
- A **bad model promoted and rolled back** with one metadata change each, watched live through the endpoint.
- **Nothing left running.** The last section deletes everything the lab created and proves it.

> **Cost and teardown promise.** Everything runs on free tiers: Colab's free CPU runtime, a free Hugging Face account, a free Gradio share link and a local MLflow store. Expected spend: **$0**. Section 10 deletes every resource this lab created and verifies each deletion.

> **Heads-up:** for about an hour this lab puts a model repository and a web app **on the public internet, under your own Hugging Face account**. Section 10 takes them down again.

**Time:** 60-90 minutes. Use a Colab **CPU** runtime; no GPU is needed.

### What this costs

| Piece | Where it runs | Tier | Cost |
|---|---|---|---|
| Notebook, training, MLflow | your Colab VM | Colab free CPU runtime | $0 |
| Model registry | Hugging Face Hub, public model repo (~0.3 MB, deleted after ~1 hour) | free account; free accounts get best-effort free *public* storage ([storage limits](https://huggingface.co/docs/hub/storage-limits)) | $0 |
| Web app + public URL | a process in your Colab VM, published through a Gradio share link | free ([Gradio share links](https://www.gradio.app/guides/sharing-your-app) are free, public, and expire after one week) | $0 |

Nothing in this lab asks for a payment method, and a free Hugging Face account without one cannot be billed.

**What we deliberately did not use.** A managed Hugging Face Space would be the obvious host for the app. We checked on 23 September 2026: hosting a Gradio Space on the free *CPU Basic* hardware now requires a **PRO** subscription (a free account gets `402 Payment Required`). Free accounts may only run Gradio Spaces on *ZeroGPU*, which this lab does not use.

**The $30 programme budget is contingency only.** If it were ever needed, it would buy, for example, three months of **HF PRO** for one instructor account ($9/month on the [pricing page](https://huggingface.co/pricing), checked 23 September 2026), enough to keep a permanent demo of the reference model on a CPU Basic Space. Or it would buy compute units on Colab's **Pay As You Go** tier for a student whose free runtime gets throttled. Neither is on the happy path.

### Why this platform

The **registry** is a Hugging Face **model repository**, driven entirely from Python through `huggingface_hub.HfApi` with your own token. The **web app** is an ordinary `app.py`. It runs as a separate process in your Colab VM, loads the model from the registry, and is published through a Gradio share link. This combination is free, needs no credit card and no cloud console, and works the same in Colab and locally. Teardown is one API call plus stopping one process, which is what makes it safe to hand to 30 people at once.

> At work, when the thing you are deploying is not a demo, you would reach for a managed platform such as [Modal](https://modal.com). It gives you serverless containers from Python, with a stable URL, autoscaling and a proper container build. Everything else in this notebook stays the same.

## Setup

Five things happen here, in order:
1. install the pinned libraries;
2. record the environment;
3. read your Hugging Face token;
4. check whose token it is;
5. name the one repository this run will create.

**Why pin?** The deck's *Pinning the Environment* slide calls an unpinned `pip install` "*broken by definition*". Colab already ships `numpy`, `scikit-learn`, `pandas`, `gradio`, `datasets` and `huggingface_hub`. We pin the libraries this lab adds, and pin the Colab-shipped ones whose APIs we call to the versions this notebook was tested with. We deliberately do **not** force-reinstall `numpy` or `scikit-learn` over Colab's copies, because that forces a runtime restart. Whatever versions you have are **recorded** below instead, and written into the serving environment's `requirements.txt` in section 6.

In [ ]:
# The MLOps libraries this lab adds, plus Colab-shipped libraries whose APIs we rely on,
# pinned to the versions this notebook was tested with (Colab runtime, September 2026).
PINNED = [
    "mlflow-skinny==3.16.1",   # MLflow's tracking client only: no server, no UI dependencies
    "optuna==5.0.0",
    "skops==0.15.0",
    "huggingface_hub==1.31.0",
    "datasets==4.8.5",
    "gradio==6.27.0",
    "gradio_client==2.7.0",
]
# Unpinned on purpose: Colab already has them, so pip leaves them alone; elsewhere they get installed.
PREINSTALLED = ["numpy", "scipy", "pandas", "scikit-learn", "matplotlib", "pillow", "requests"]
_pkgs = " ".join(PINNED + PREINSTALLED)
%pip install -q {_pkgs}
# pip may print "you may need to restart the kernel": ignore it. Nothing this lab uses was imported yet.

In [ ]:
import os, sys, json, re, time, random, platform, hashlib, secrets, shutil, signal, subprocess, warnings, logging
import datetime as dt
from pathlib import Path
from importlib.metadata import version

# Quieten libraries that narrate at INFO level. Errors still show.
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
for noisy in ("mlflow", "alembic", "httpx", "optuna", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

SEED = 42                 # the one seed: every random_state below is this value
random.seed(SEED)
np.random.seed(SEED)      # the legacy global RNG; scikit-learn still gets random_state explicitly everywhere
# PYTHONHASHSEED only takes effect if set before the interpreter starts (the deck's seeding slide).
# Nothing here depends on hash order, so we just record it.

IN_COLAB = "google.colab" in sys.modules
WORK = Path("mlops_lab_work")            # everything this notebook writes locally lives here
for sub in ("", "serving", "traffic"):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK.resolve()))  # so that `import preprocess` finds our module

LIBS = ["numpy", "scipy", "pandas", "scikit-learn", "pillow", "datasets", "huggingface_hub",
        "mlflow-skinny", "optuna", "skops", "gradio", "gradio_client", "requests", "matplotlib"]
ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "colab": IN_COLAB,
    "cpu_count": os.cpu_count(),
    "seed": SEED,
    "pythonhashseed": os.environ.get("PYTHONHASHSEED", "unset"),
    **{lib: version(lib) for lib in LIBS},
}
ENV_HASH = hashlib.sha256(json.dumps(ENV, sort_keys=True).encode()).hexdigest()[:12]

print(f"Environment fingerprint {ENV_HASH}")
for key, value in ENV.items():
    print(f"  {key:<16} {value}")

Every network call in this lab goes through the two helpers below:
- every call has a **timeout**;
- **transient** failures (timeouts, dropped connections, HTTP 429 and 5xx) are **retried** with backoff;
- a final failure **stops the cell with one sentence** saying what to check, instead of a 40-line traceback. Conference wifi happens.

In [ ]:
import httpx
import requests
from huggingface_hub import HfApi, set_client_factory
from huggingface_hub.errors import RepositoryNotFoundError

# huggingface_hub's default HTTP client has NO timeout: on bad wifi a call can hang forever.
set_client_factory(lambda: httpx.Client(follow_redirects=True, timeout=httpx.Timeout(60.0, connect=15.0)))

class LabStop(Exception):
    # Raised when the lab cannot continue. Shown as one readable message, not a traceback.
    pass

def stop(message):
    raise LabStop(message)

def _show_lab_stop(shell, etype, evalue, tb, tb_offset=None):
    print(f"\n[STOP] {evalue}\n", file=sys.stderr)
    return [f"LabStop: {evalue}"]

try:
    get_ipython().set_custom_exc((LabStop,), _show_lab_stop)
except NameError:
    pass   # not running under IPython

RETRYABLE = (httpx.TimeoutException, httpx.NetworkError, httpx.RemoteProtocolError,
             requests.ConnectionError, requests.Timeout, TimeoutError, ConnectionError)

def with_retry(what, fn, attempts=4, first_wait=2.0,
               hint="your internet connection, and https://status.huggingface.co"):
    # Run fn(). Retry transient network failures with exponential backoff; stop readably otherwise.
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            transient = isinstance(e, RETRYABLE) or status in (429, 500, 502, 503, 504)
            if not transient or attempt == attempts:
                first_line = (str(e).strip().splitlines() or [""])[0][:300]
                stop(f"{what} failed ({type(e).__name__}: {first_line}). Check: {hint}.")
            wait = first_wait * 2 ** (attempt - 1)
            print(f"  {what}: {type(e).__name__}, attempt {attempt}/{attempts}; retrying in {wait:.0f}s")
            time.sleep(wait)

### Your Hugging Face token

The lab creates, and later deletes, one model repository under **your** account, so it needs a token with **write** access. Every student uses their own token, and nothing is shared.

**Create one (2 minutes):**
1. Sign in at [huggingface.co](https://huggingface.co). A free account is enough; sign up if you need to.
2. Open [**Settings → Access Tokens**](https://huggingface.co/settings/tokens) and click **Create new token**.
3. Choose the token type **Write**, name it `mlops-lab`, click **Create token** and copy it. It is shown only once.
4. In Colab, click the **key icon (Secrets)** in the left sidebar → **Add new secret**. Name it `HF_TOKEN`, paste the value, and switch **Notebook access** on.

The next cell looks for the token in three places, in order:
1. that Colab secret;
2. an `HF_TOKEN` environment variable (for local runs);
3. a hidden prompt, where you paste it.

The token is never printed, never written to a cell output and never saved to disk. If you have no account, press Enter at the prompt to continue in **Plan B** (see the appendix).

> **Fine-grained tokens** are the production-grade choice (least privilege, the same idea as on the deck's governance slides). They work here if you grant **write access to contents/settings of repos under your personal namespace** (`repo.write` in the API), plus the same on the organisation if an instructor sets `HF_ORG`. The cell checks that this permission is present.

> **When you finish the lab, revoke the token:** Settings → Access Tokens → **Manage** next to the token → **Delete**. A token you are not using is a liability.

In [ ]:
from getpass import getpass

def read_hf_token():
    # Colab secret -> environment variable -> hidden prompt. Returns (token, where_it_came_from).
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token.strip(), "Colab secret HF_TOKEN"
    except Exception:
        pass   # not in Colab, no such secret, or notebook access not granted
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"].strip(), "environment variable HF_TOKEN"
    try:
        return getpass("Paste your Hugging Face WRITE token (hidden; press Enter alone for Plan B): ").strip(), "hidden prompt"
    except Exception:
        return "", "nowhere (this frontend cannot prompt)"

_hf_token, _token_source = read_hf_token()
PLAN_B = not _hf_token
if PLAN_B:
    api, USER, ROLE, _access = None, "local", None, {}
    print("No token: continuing in Plan B. Nothing will be published; see the appendix for which sections to run.")
else:
    api = HfApi(token=_hf_token)
    me = with_retry("Checking your token (whoami)", api.whoami,
                    hint="that you copied the whole token and have not deleted it")
    USER = me["name"]
    _access = me.get("auth", {}).get("accessToken", {})
    ROLE = _access.get("role", "unknown")
    print(f"Token OK (read from the {_token_source}).")
    print(f"  it belongs to : {USER}")
    print(f"  token role    : {ROLE}")
    if ROLE == "read":
        stop("This token is read-only. Create a token of type Write (steps above) and re-run this cell.")

### Name what this run will create

Every repository this lab creates is named `mlops-lab-<run id>-digits`, in **your own namespace**. The run id is the date plus four random characters, for example `260923-k3x9`. So two students, or one student running the lab twice, can never collide.

The run id is saved in `mlops_lab_work/run_state.json`. If a cell fails and you re-run from the top, you get the **same** names back instead of leaking a second repository.

In [ ]:
HF_ORG = None   # Instructors: set this to an organisation name to put every student's repo there instead.

def slug(text):
    return re.sub(r"[^a-z0-9-]+", "-", text.lower()).strip("-")

def new_run_id():
    return f"{OWNER_PREFIX}{dt.date.today():%y%m%d}-{''.join(secrets.choice('abcdefghijklmnopqrstuvwxyz0123456789') for _ in range(4))}"

NAMESPACE = HF_ORG or USER
OWNER_PREFIX = f"{slug(USER)}-" if HF_ORG else ""   # in a shared organisation, your username keeps names apart
STATE_FILE = WORK / "run_state.json"
state = json.loads(STATE_FILE.read_text(encoding="utf-8")) if STATE_FILE.exists() else {}
REUSED = state.get("namespace") == NAMESPACE and bool(state.get("run_id"))
RUN_ID = state["run_id"] if REUSED else new_run_id()
MODEL_REPO_ID = f"{NAMESPACE}/mlops-lab-{RUN_ID}-digits"

if not PLAN_B:
    if ROLE == "fineGrained":
        scoped = _access.get("fineGrained", {}).get("scoped", [])
        perms = next((s.get("permissions", []) for s in scoped if s.get("entity", {}).get("name") == NAMESPACE), [])
        if "repo.write" not in perms:
            stop(f"Your fine-grained token cannot create repos in '{NAMESPACE}'. Edit the token and grant "
                 "write access to repos there (repo.write), or use a Write token.")
    # A brand-new run id must name a repo that does not exist yet: collisions are impossible, not just unlikely.
    while not REUSED and with_retry("Checking the name is free", lambda: api.repo_exists(MODEL_REPO_ID)):
        RUN_ID = new_run_id()
        MODEL_REPO_ID = f"{NAMESPACE}/mlops-lab-{RUN_ID}-digits"
STATE_FILE.write_text(json.dumps({"namespace": NAMESPACE, "run_id": RUN_ID}), encoding="utf-8")

# The exact shape of every repo name this notebook can create. The teardown sweeper deletes nothing else.
LAB_REPO_PATTERN = re.compile(rf"^{re.escape(NAMESPACE)}/mlops-lab-{re.escape(OWNER_PREFIX)}\d{{6}}-[a-z0-9]{{4}}-digits$")
assert LAB_REPO_PATTERN.fullmatch(MODEL_REPO_ID)

print(f"run id      : {RUN_ID}{'   (re-used from ' + str(STATE_FILE) + ')' if REUSED else ''}")
print(f"namespace   : {NAMESPACE}")
print(f"model repo  : {MODEL_REPO_ID}   (created in section 5, deleted in section 10)" if not PLAN_B
      else "model repo  : none (Plan B publishes nothing to the Hub)")

# Content

## 1. Data: a pinned snapshot

> **From the deck, Reproducibility → *Versioning the Data*:** "*Never version a mutable database table by referring to 'the customers table'. Refer to a snapshot, or to a query plus an as-of timestamp.*"

A dataset on the Hugging Face Hub is a git repository, so every snapshot has a commit sha: data versioning for free. We **pin** one. `DATASET_REVISION` below is the commit this lab was written against. Loading "whatever `main` is today" would be the mutable-table mistake. The cell also shows where `main` is now, so you can see whether the world moved.

In [ ]:
from datasets import load_dataset

DATASET_ID = "ylecun/mnist"
DATASET_REVISION = "77f3279092a1c1579b2250db8eafed0ad422088c"   # pinned snapshot (last changed 2024-08-08)

head = with_retry("Looking up the dataset", lambda: (api or HfApi()).dataset_info(DATASET_ID).sha)
print(f"pinned revision : {DATASET_REVISION}")
print(f"current main    : {head}   "
      f"({'unchanged' if head == DATASET_REVISION else 'MOVED since this lab was written; we still train on the pinned snapshot'})")

ds = with_retry("Downloading MNIST", lambda: load_dataset(DATASET_ID, revision=DATASET_REVISION))
print(ds)

### Subsampling: legitimate here, illegitimate elsewhere

We train on a stratified **10,000** of the 60,000 training images and hold out a stratified **2,000** of the 10,000 test images. That keeps each training run to a few seconds on a Colab CPU, which is what lets us afford 80 tuning trials.

**Legitimate here**, because:
- the goal is the loop, not the leaderboard;
- MNIST is i.i.d. and highly redundant;
- the sample is stratified, so every class keeps its share;
- the sampled indices are hashed below, so anyone can rebuild the exact same sample.

**Illegitimate elsewhere**, if any of these is true:
- you report the number as "MNIST accuracy";
- the data has a rare class or slice that matters (fraud, a rare disease, a minority dialect), because subsampling shrinks exactly the tail you care about;
- the data is ordered in time, because a random sample leaks the future into training;
- the sample was never recorded. Then it is not a subsample, it is an irreproducible accident.

The other 8,000 test images become the **traffic pool**: the images we send to the live model in section 8.

In [ ]:
from sklearn.model_selection import train_test_split

y_train_all = np.array(ds["train"]["label"])
y_test_all = np.array(ds["test"]["label"])

# Stratified, seeded subsamples: indices into the pinned snapshot.
train_idx, _ = train_test_split(np.arange(len(y_train_all)), train_size=10_000, stratify=y_train_all, random_state=SEED)
test_idx, pool_idx = train_test_split(np.arange(len(y_test_all)), train_size=2_000, stratify=y_test_all, random_state=SEED)
# 10k training images = 8k to fit + 2k to validate while tuning. The test set is touched once, at the end.
fit_idx, val_idx = train_test_split(train_idx, test_size=2_000, stratify=y_train_all[train_idx], random_state=SEED)
fit_idx, val_idx, test_idx, pool_idx = map(np.sort, (fit_idx, val_idx, test_idx, pool_idx))

SPLIT_HASH = hashlib.sha256(json.dumps({
    "dataset": DATASET_ID, "revision": DATASET_REVISION,
    "fit": fit_idx.tolist(), "val": val_idx.tolist(), "test": test_idx.tolist(),
}).encode()).hexdigest()[:16]

print(f"fit {len(fit_idx)} | val {len(val_idx)} | test {len(test_idx)} | traffic pool {len(pool_idx)}")
print(f"SPLIT_HASH = {SPLIT_HASH}")

> **Try it:** compare your `SPLIT_HASH` with your neighbour's. Same seed plus same snapshot should give the same hash on a different machine. That is level 2, *reproducible*, on the deck's *"It Worked Last Tuesday"* slide: a colleague, a different machine, the same number.

### The preprocessing contract: one file, used by training *and* serving

> **From the deck, Packaging & Deployment → *"Ship the Model" - Ship What, Exactly?*:** "*Forgetting the **fitted** preprocessing is the classic production bug.*" The slide's rule is to serialise the **whole pipeline**, "*one artifact, one `predict`, no room for skew*."

The model will see 784 numbers per image. People will send it photos, phone drawings, white-on-black and black-on-white. Turning *an image* into *those 784 numbers* is preprocessing too.

If that conversion lives only in this notebook, the serving code will re-implement it slightly differently. That is **training/serving skew**, and it fails silently: the service stays up and returns confident, wrong answers.

So the conversion lives in exactly **one** file, `preprocess.py`:
- the notebook imports it to build the training set;
- the same file is uploaded to the registry **in the same commit as the model**;
- the serving app downloads and imports that copy.

Change one line in it and you have a new model version, whether or not you retrained.

The function mimics how MNIST itself was built: light ink on black, the digit scaled into a 20×20 box and centred by its centre of mass in a 28×28 frame.

In [ ]:
%%writefile mlops_lab_work/preprocess.py
"""preprocess.py - the ONE place an image becomes model input.

Imported by the training notebook AND by the serving app, and uploaded to the model
registry next to the model, in the same commit. Change anything here and you have made
a new model version, whether or not you retrained.
"""
import hashlib

import numpy as np
from PIL import Image

PREPROCESS_VERSION = "1.0"
SIDE = 28            # MNIST frame
BOX = 20             # MNIST digits are scaled to fit a 20x20 box ...
INK_THRESHOLD = 30   # ... after cropping to pixels brighter than this (0-255)
N_FEATURES = SIDE * SIDE


def _as_pil(image):
    """Accept a PIL image, a numpy array, a file path, or the dict a Gradio Sketchpad
    returns ({'background', 'layers', 'composite'})."""
    if isinstance(image, dict):
        image = image.get("composite")
    if image is None:
        raise ValueError("no image received")
    if isinstance(image, Image.Image):
        return image
    if isinstance(image, (str, bytes)) or hasattr(image, "__fspath__"):
        with Image.open(image) as im:
            im.load()
            return im.copy()
    arr = np.asarray(image)
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def to_model_input(image) -> np.ndarray:
    """One image of one handwritten digit -> float32 vector of 784 values in [0, 1],
    laid out like MNIST: light ink on black, digit fitted into a 20x20 box and centred
    by its centre of mass in a 28x28 frame."""
    img = _as_pil(image)

    # Transparent pixels (a drawing canvas) mean paper, i.e. white.
    if img.mode in ("RGBA", "LA", "PA") or (img.mode == "P" and "transparency" in img.info):
        img = img.convert("RGBA")
        paper = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(paper, img)
    a = np.asarray(img.convert("L"), dtype=np.uint8)

    # MNIST is light ink on a dark background. If the border is bright, flip it.
    border = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]])
    if border.mean() > 127:
        a = 255 - a

    ink = a > INK_THRESHOLD
    if not ink.any():
        return np.zeros(N_FEATURES, dtype=np.float32)   # blank input: nothing to centre
    rows, cols = np.where(ink)
    crop = a[rows.min():rows.max() + 1, cols.min():cols.max() + 1]

    # Fit the longer side to BOX pixels, keeping the aspect ratio.
    h, w = crop.shape
    scale = BOX / max(h, w)
    nh, nw = max(1, round(h * scale)), max(1, round(w * scale))
    small = np.asarray(Image.fromarray(crop).resize((nw, nh), Image.Resampling.BILINEAR), dtype=np.float32)
    if small.max() > 0:
        small *= 255.0 / small.max()    # thin strokes lose brightness when shrunk; restore the peak

    # Paste so that the centre of mass lands in the middle of the 28x28 frame.
    ys, xs = np.indices(small.shape)
    mass = small.sum()
    cy, cx = (ys * small).sum() / mass, (xs * small).sum() / mass
    top = int(np.clip(round(SIDE / 2 - cy), 0, SIDE - nh))
    left = int(np.clip(round(SIDE / 2 - cx), 0, SIDE - nw))
    canvas = np.zeros((SIDE, SIDE), dtype=np.float32)
    canvas[top:top + nh, left:left + nw] = small

    return (canvas / 255.0).astype(np.float32).ravel()


def to_model_batch(images) -> np.ndarray:
    return np.stack([to_model_input(im) for im in images])


def feature_hash(x) -> str:
    """Fingerprint of one input AS THE MODEL SAW IT (after preprocessing)."""
    return hashlib.sha256(np.ascontiguousarray(x, dtype=np.float32).tobytes()).hexdigest()[:16]

In [ ]:
import importlib
from PIL import Image, ImageDraw
import preprocess as pp
importlib.reload(pp)   # picks up your edits if you change the file above and re-run

# A "phone drawing": dark ink on white paper, a big canvas, off-centre. Nothing like MNIST.
phone = Image.new("RGB", (400, 300), "white")
ImageDraw.Draw(phone).line([(230, 60), (300, 60), (255, 250)], fill="black", width=22)

examples = [(ds["train"][int(i)]["image"], f"MNIST, label {y_train_all[i]}") for i in fit_idx[:3]]
examples.append((phone, "a phone-style drawing"))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for col, (img, title) in enumerate(examples):
    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(title, fontsize=9)
    axes[1, col].imshow(pp.to_model_input(img).reshape(28, 28), cmap="gray")
    axes[1, col].set_title("what the model sees", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def features(split, idx):
    return pp.to_model_batch(ds[split].select(idx)["image"])

t0 = time.perf_counter()
X_fit, X_val, X_test = features("train", fit_idx), features("train", val_idx), features("test", test_idx)
y_fit, y_val, y_test = y_train_all[fit_idx], y_train_all[val_idx], y_test_all[test_idx]
test_images = list(ds["test"].select(test_idx)["image"])   # the raw images behind X_test, used again later

PREPROCESS_SHA = hashlib.sha256((WORK / "preprocess.py").read_bytes()).hexdigest()[:16]
print(f"X_fit {X_fit.shape} | X_val {X_val.shape} | X_test {X_test.shape}   ({time.perf_counter() - t0:.1f}s)")
print(f"preprocess.py sha256 {PREPROCESS_SHA} | training matrix hash {hashlib.sha256(X_fit.tobytes()).hexdigest()[:16]}")

## 2. A baseline, tracked

> **From the deck, Experiment Tracking → *The Data Model: What a "Run" Contains*:** an **experiment** groups **runs**; each run records **parameters**, **metrics** (per step, not only the final number), **artifacts** and **tags**. "*Log the inputs (commit, config, data hash) as parameters. Without them a run is a number without provenance.*"

We use **MLflow** with a **local store**: one SQLite file holds the metadata and one folder holds the artifacts. There is no server, no tunnel and no account. It is the deck's *Where Does the Tracking Data Actually Live?* slide in miniature: metadata in a database, artifacts in file storage.

> MLflow 3.x changed its default from a folder of files (`./mlruns`) to SQLite, and the old file store now refuses to start unless you opt in. That is a live example of why you pin library versions: the same code, one release apart, behaves differently.

And the deck's **baseline rule**: "*Before any model goes to production, log a run for the dumbest possible baseline.*" So the first run always predicts the majority class.

In [ ]:
import mlflow
from mlflow.entities import Metric
from mlflow.tracking import MlflowClient
logging.getLogger("mlflow").setLevel(logging.WARNING)   # mlflow resets its own logger on import

MLFLOW_URI = f"sqlite:///{(WORK / 'mlflow.db').resolve().as_posix()}"
mlflow.set_tracking_uri(MLFLOW_URI)
EXPERIMENT = "mnist-mlops-lab"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(WORK / "mlartifacts").resolve().as_uri())
mlflow.set_experiment(EXPERIMENT)
tracker = MlflowClient()

# Provenance that goes on EVERY run: which data, which code, which environment, which seed.
PROVENANCE = {
    "data.dataset": DATASET_ID, "data.revision": DATASET_REVISION, "data.split_hash": SPLIT_HASH,
    "data.n_fit": len(fit_idx), "data.n_val": len(val_idx),
    "code.preprocess_sha256": PREPROCESS_SHA, "env.hash": ENV_HASH, "seed": SEED, "lab.run_id": RUN_ID,
}
print("tracking store:", MLFLOW_URI)

In [ ]:
import skops.io as sio
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # we cap the epochs on purpose

def make_pipeline(pca_components=64, hidden_units=128, learning_rate_init=1e-3, alpha=1e-4, max_iter=20):
    # The whole model as ONE object: fitted PCA, fitted scaler, classifier.
    return Pipeline([
        ("pca", PCA(n_components=pca_components, random_state=SEED)),
        ("scale", StandardScaler()),     # the fitted normalisation step: its means and scales ship with the model
        ("clf", MLPClassifier(hidden_layer_sizes=(hidden_units,), learning_rate_init=learning_rate_init,
                              alpha=alpha, batch_size=128, max_iter=max_iter, random_state=SEED)),
    ])

def tracked_fit(name, model, params):
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({**PROVENANCE, **params})
        t0 = time.perf_counter()
        model.fit(X_fit, y_fit)
        fit_s = time.perf_counter() - t0
        val_acc = accuracy_score(y_val, model.predict(X_val))
        mlflow.log_metrics({"val_accuracy": val_acc, "fit_seconds": fit_s})
        mlflow.log_dict(ENV, "environment.json")
        path = WORK / f"{name}.skops"
        sio.dump(model, path)
        mlflow.log_artifact(str(path))
    print(f"{name:<22} val_accuracy={val_acc:.4f}   fit={fit_s:.1f}s   run {run.info.run_id[:12]}")

tracked_fit("baseline-majority", DummyClassifier(strategy="most_frequent"), {"model": "majority class"})
tracked_fit("baseline-default-mlp", make_pipeline(),
            {"model": "pca+scale+mlp", "pca_components": 64, "hidden_units": 128,
             "learning_rate_init": 1e-3, "alpha": 1e-4, "max_iter": 20})

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["start_time ASC"])
runs[["tags.mlflow.runName", "metrics.val_accuracy", "metrics.fit_seconds",
      "params.data.revision", "params.data.split_hash", "params.env.hash", "run_id"]]

> To browse the same store in the MLflow UI on your own machine, install the full `mlflow` package and run `mlflow ui --backend-store-uri sqlite:///mlops_lab_work/mlflow.db`. We don't launch it from Colab, because it would need a tunnel of its own, and `search_runs` returns the same data as a DataFrame.

## 3. Tuning, with pruning, and evidence

> **From the deck, Hyperparameter Optimization:** TPE models "*what good configurations look like*" instead of sampling blindly, and pruning stops "*a trial that is obviously hopeless at epoch 2*" from running "*to epoch 50*". Then comes the claim we are going to test: "*TPE beats random search at equal time - but the two curves that collapse almost immediately are the pruned ones. **Pruning is the bigger win.***" (Akiba et al., [Optuna, KDD 2019](https://arxiv.org/abs/1907.10902v1), Fig. 11a.)

What we tune:
- The **pipeline**, not only the model, as the deck's search-space slide says: PCA size, hidden units, learning rate and L2 strength (the last two on a log scale).
- The classifier is an `MLPClassifier` precisely because it trains in **epochs**. After each epoch a trial reports its validation accuracy, and the **median pruner** stops it if it is below the median of earlier trials at the same epoch. A `LogisticRegression` has no intermediate values worth reporting, so there would be nothing to prune.

To test the deck's claim rather than repeat it, we spend the same 20-trial budget four ways: TPE or random sampling, each with and without pruning. **Every trial, including the pruned ones, is logged to MLflow** ("*keep failed runs, tagged. They are evidence*"). The objective uses the **validation** split only; the test set stays untouched ("*Guard the test set*"). The whole cell takes 1-2 minutes on Colab.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS, MAX_EPOCHS = 20, 20
CLASSES = np.arange(10)
_feature_cache = {}

def pca_features(k):
    # PCA + scaling depend only on k, so fit them once per k and share them across trials.
    if k not in _feature_cache:
        pre = Pipeline([("pca", PCA(n_components=k, random_state=SEED)), ("scale", StandardScaler())]).fit(X_fit)
        _feature_cache[k] = (pre.transform(X_fit), pre.transform(X_val))
    return _feature_cache[k]

def make_objective(arm):
    def objective(trial):
        params = {
            "pca_components": trial.suggest_categorical("pca_components", [32, 64, 128]),
            "hidden_units": trial.suggest_categorical("hidden_units", [64, 128, 256]),
            "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 3e-1, log=True),
            "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        }
        F_fit, F_val = pca_features(params["pca_components"])
        clf = MLPClassifier(hidden_layer_sizes=(params["hidden_units"],), alpha=params["alpha"],
                            learning_rate_init=params["learning_rate_init"], batch_size=128, random_state=SEED)
        curve, pruned = [], False
        with mlflow.start_run(run_name=f"{arm}-trial-{trial.number:02d}", nested=True) as run:
            mlflow.log_params({**params, "arm": arm})
            for epoch in range(1, MAX_EPOCHS + 1):
                clf.partial_fit(F_fit, y_fit, classes=CLASSES)
                curve.append(accuracy_score(y_val, clf.predict(F_val)))
                trial.report(curve[-1], epoch)
                if trial.should_prune():
                    pruned = True
                    break
            # The learning curve in one write, then the outcome. Pruned runs are evidence too.
            now_ms = int(time.time() * 1000)
            tracker.log_batch(run.info.run_id, metrics=[Metric("val_accuracy", a, now_ms, step)
                                                        for step, a in enumerate(curve, 1)])
            mlflow.set_tags({"trial_state": "PRUNED" if pruned else "COMPLETE", "epochs_run": len(curve)})
        trial.set_user_attr("epochs_run", len(curve))
        if pruned:
            raise optuna.TrialPruned()
        return curve[-1]
    return objective

def median_pruner():
    return optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

ARMS = {   # the same budget spent four ways
    "tpe+pruning":    (optuna.samplers.TPESampler(seed=SEED, n_startup_trials=6), median_pruner()),
    "tpe":            (optuna.samplers.TPESampler(seed=SEED, n_startup_trials=6), optuna.pruners.NopPruner()),
    "random+pruning": (optuna.samplers.RandomSampler(seed=SEED), median_pruner()),
    "random":         (optuna.samplers.RandomSampler(seed=SEED), optuna.pruners.NopPruner()),
}
for k in (32, 64, 128):
    pca_features(k)          # pay for the features up front, so no arm is charged for them

studies, finished_at, summary = {}, {}, []
for arm, (sampler, pruner) in ARMS.items():
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name=arm)
    stamps, t0 = [], time.perf_counter()
    with mlflow.start_run(run_name=f"hpo-{arm}"):
        mlflow.log_params({**PROVENANCE, "arm": arm, "n_trials": N_TRIALS, "max_epochs": MAX_EPOCHS})
        study.optimize(make_objective(arm), n_trials=N_TRIALS,
                       callbacks=[lambda s, t: stamps.append(time.perf_counter() - t0)])
        epochs = sum(t.user_attrs.get("epochs_run", 0) for t in study.trials)
        n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
        row = {"arm": arm, "best val accuracy": round(study.best_value, 4), "trials pruned": n_pruned,
               "epochs trained": f"{epochs} / {N_TRIALS * MAX_EPOCHS}", "wall seconds": round(stamps[-1], 1)}
        mlflow.log_metrics({"best_val_accuracy": study.best_value, "wall_seconds": stamps[-1],
                            "epochs_trained": epochs, "trials_pruned": n_pruned})
    studies[arm], finished_at[arm] = study, stamps
    summary.append(row)
    print(f"{arm:<15} best {study.best_value:.4f} | pruned {n_pruned:2d}/{N_TRIALS} | "
          f"epochs {epochs:3d}/{N_TRIALS * MAX_EPOCHS} | {stamps[-1]:5.1f}s")

pd.DataFrame(summary).set_index("arm")

The chart puts **elapsed wall-clock time** on the x-axis, as the deck's pruning figure does, because time is the axis that costs money. Each line is the best validation accuracy found so far by one arm. The cell after it turns the chart into one number per effect.

In [ ]:
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, CRITICAL = "#2a78d6", "#eb6834", "#d03b3b"

def tidy(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(INK_2)
    ax.tick_params(colors=INK_2, labelsize=9)
    ax.set_axisbelow(True)

def best_so_far(arm):
    xs, ys, best = [], [], None
    for trial, stamp in zip(studies[arm].trials, finished_at[arm]):
        if trial.state == optuna.trial.TrialState.COMPLETE and (best is None or trial.value > best):
            best = trial.value
        if best is not None:
            xs.append(stamp)
            ys.append(best)
    return xs, ys

LOOK = {"tpe+pruning": (BLUE, "-"), "tpe": (BLUE, "--"), "random+pruning": (ORANGE, "-"), "random": (ORANGE, "--")}
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for arm in ARMS:
    xs, ys = best_so_far(arm)
    color, style = LOOK[arm]
    ax.step(xs, ys, where="post", color=color, linestyle=style, linewidth=2, label=arm)
    ax.plot(xs[-1], ys[-1], "o", color=color, markersize=7, markeredgecolor="white", markeredgewidth=2)
ax.set_xlabel("elapsed seconds (wall clock)", color=INK_2)
ax.set_ylabel("best validation accuracy so far", color=INK_2)
ax.set_title("Same 20-trial budget, four ways (dot = arm finished)", color=INK, fontsize=11, loc="left")
ax.legend(frameon=False, fontsize=9, labelcolor=INK_2)
tidy(ax)
plt.tight_layout()
plt.show()

In [ ]:
target = max(s.best_value for s in studies.values()) - 0.005   # "within half a point of the best anyone found"

def seconds_to_target(arm):
    xs, ys = best_so_far(arm)
    return next((x for x, y in zip(xs, ys) if y >= target), float("inf"))

reach = {arm: seconds_to_target(arm) for arm in ARMS}
epochs_budget = N_TRIALS * MAX_EPOCHS
epochs_pruned = sum(t.user_attrs["epochs_run"] for t in studies["tpe+pruning"].trials)
print(f"target: validation accuracy >= {target:.4f}")
for arm, s in reach.items():
    print(f"  {arm:<15} reached it after {s:5.1f}s" if s < float("inf") else f"  {arm:<15} never reached it")
print(f"\nCompute saved by pruning (TPE arm): {1 - epochs_pruned / epochs_budget:.0%} of the epochs "
      f"({epochs_pruned} of {epochs_budget}) and {finished_at['tpe'][-1] - finished_at['tpe+pruning'][-1]:.1f}s of wall clock.")

def speedup(slow, fast):
    if reach[fast] == float("inf"):
        return None
    return reach[slow] / reach[fast]

from_pruning = speedup("tpe", "tpe+pruning")   # same sampler, pruner switched on
from_tpe = speedup("random+pruning", "tpe+pruning")   # same pruner, smarter sampler
print(f"Time-to-target speed-up from pruning : {'n/a' if from_pruning is None else f'{from_pruning:.1f}x'}")
print(f"Time-to-target speed-up from TPE     : {'n/a' if from_tpe is None else f'{from_tpe:.1f}x'}")
if from_pruning and from_tpe:
    bigger = "pruning" if from_pruning > from_tpe else "TPE"
    print(f"On this run, the bigger win was {bigger}.")

**Read your own output, not ours.** Twenty trials with one seed is a small, noisy experiment, and the result depends on the run. Keep two things apart:

- **The compute saved by pruning** is robust. It is roughly the share of trials that were hopeless, times the epochs they did not train, and it grows with the cost of an epoch.
- **Which effect wins time-to-target** can go either way at this scale. The deck's figure comes from a deep network where one trial costs minutes. Here a trial costs about a second and the search space is easy, so a random sampler also lands on good configurations quickly, and a speed-up of "1.0x" simply means both arms found the target on the same trial.

If your run disagrees with the deck, that is a result worth writing down, not a mistake. It is exactly what the deck's *Comparing Runs Honestly* slide asks for.

> **Try it:** change `N_TRIALS` to 40 or `MAX_EPOCHS` to 40 and re-run the section. Which number moves, the compute saved or the speed-up?

In [ ]:
best = studies["tpe+pruning"].best_trial
BEST_PARAMS = dict(best.params)
print(f"selected: trial {best.number} of the tpe+pruning arm, val accuracy {best.value:.4f}")
print(json.dumps(BEST_PARAMS, indent=1))

hpo = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="params.arm = 'tpe+pruning'")
hpo = hpo[hpo["tags.trial_state"].notna()]
print(hpo["tags.trial_state"].value_counts().to_string())
hpo.sort_values("metrics.val_accuracy", ascending=False)[
    ["tags.mlflow.runName", "tags.trial_state", "tags.epochs_run", "metrics.val_accuracy",
     "params.pca_components", "params.hidden_units", "params.learning_rate_init", "params.alpha"]].head(8)

## 4. Package: one artifact, one contract

> **From the deck, *"Ship the Model" - Ship What, Exactly?*:** five boxes: the *learned parameters*, the *fitted preprocessing*, the *inference code*, the *environment*, and the *metadata* (schema, metrics, lineage, owner). This is where each of them ends up:

| Deck box | In this lab | File |
|---|---|---|
| Learned parameters | the MLP's weights | `model.skops` |
| Fitted preprocessing | PCA + `StandardScaler`, inside the same `Pipeline` | `model.skops` |
| Image → numbers | the preprocessing contract | `preprocess.py`, same commit |
| Inference code | the serving app (section 6) | `serving/app.py` |
| Environment | exact library versions | `environment.json`, `serving/requirements.txt` |
| Metadata | input contract, classes, metrics, data revision, owner, timestamp, reviewed types | `schema.json`, `README.md` (the model card) |

First we refit the chosen configuration on **train + validation**, all 10,000 images, because the validation split has done its job. Then we touch the **test set once**.

In [ ]:
X_train, y_train = np.concatenate([X_fit, X_val]), np.concatenate([y_fit, y_val])

def evaluate(model):
    pred = model.predict(X_test)
    per_class = {int(c): round(float((pred[y_test == c] == c).mean()), 4) for c in CLASSES}
    return float(accuracy_score(y_test, pred)), per_class

model_v1 = make_pipeline(**BEST_PARAMS, max_iter=MAX_EPOCHS)
with mlflow.start_run(run_name="v1-final") as run:
    mlflow.log_params({**PROVENANCE, **BEST_PARAMS, "max_iter": MAX_EPOCHS, "data.n_train": len(y_train),
                       "selected_from": f"hpo-tpe+pruning trial {best.number}"})
    t0 = time.perf_counter()
    model_v1.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0
    TEST_ACC, PER_CLASS = evaluate(model_v1)
    mlflow.log_metrics({"test_accuracy": TEST_ACC, "fit_seconds": fit_s,
                        **{f"test_accuracy_class_{c}": a for c, a in PER_CLASS.items()}})
    mlflow.log_dict(ENV, "environment.json")
    V1_RUN_ID = run.info.run_id
TRAINED_AT = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

worst = min(PER_CLASS, key=PER_CLASS.get)
print(f"v1: test accuracy {TEST_ACC:.4f} on {len(y_test)} images, fit in {fit_s:.1f}s, MLflow run {V1_RUN_ID[:12]}")
print("per class:", "  ".join(f"{c}:{a:.3f}" for c, a in PER_CLASS.items()))
print(f"worst class: {worst} ({PER_CLASS[worst]:.3f}), which is what the deck's slice-test bullet is about: "
      f"the headline number hides it")

### Serialise with skops, and *read* the trust list

> **From the deck, *Serialisation Formats*:** pickle and joblib "*execute code on load*". scikit-learn's own documentation says never to load a pickle from an untrusted source, "*similarly to how you should never execute code from an untrusted source*" ([model persistence](https://scikit-learn.org/stable/model_persistence.html)). **skops** stores the estimator as data, and lets you inspect which types a file contains **before** you load it.

`get_untrusted_types` lists every type in the file that is not on skops' built-in allow-list. Passing that list straight back into `load(trusted=...)` is just pickle with extra steps. The discipline is:
1. **read the list**;
2. **decide** whether you trust each type;
3. **write the decision down**, and have every loader refuse anything that is not on it.

In [ ]:
BUNDLE_V1 = WORK / "registry_v1"
BUNDLE_V1.mkdir(exist_ok=True)
sio.dump(model_v1, BUNDLE_V1 / "model.skops")

found_types = sio.get_untrusted_types(file=BUNDLE_V1 / "model.skops")
print(f"model.skops is {(BUNDLE_V1 / 'model.skops').stat().st_size / 1024:.0f} KB. "
      f"Types in it that skops does not trust by default ({len(found_types)}):")
for t in found_types:
    print("   ", t)

You should see exactly one type: `sklearn.neural_network._stochastic_optimizers.AdamOptimizer`. It is the Adam optimiser's state, which `MLPClassifier` keeps after training so that `partial_fit` can resume. It is scikit-learn's own class, it only holds arrays, and it came out of our own training run. So we trust it, **by name**.

For contrast, here is what skops reports for a pipeline that contains a function defined in this notebook:

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def clip_to_unit(X):
    return np.clip(X, 0, 1)

custom = Pipeline([("clip", FunctionTransformer(clip_to_unit)), ("clf", DummyClassifier())]).fit(X_fit[:50], y_fit[:50])
sio.dump(custom, WORK / "demo_custom_function.skops")
print("untrusted types:", sio.get_untrusted_types(file=WORK / "demo_custom_function.skops"))
try:
    sio.load(WORK / "demo_custom_function.skops")          # no trust given
except Exception as e:
    print(f"skops refused to load it: {type(e).__name__}")

`__main__.clip_to_unit` means: *to load this file, Python must find and call some function by that name*. A pickle would have done that without asking. This is why our image preprocessing lives in `preprocess.py`, as source code you can review, while the serialised pipeline contains only scikit-learn's own estimators.

Now write the decision down:

In [ ]:
# The human decision, written down. Every loader (including the server) refuses anything else.
REVIEWED_TRUSTED_TYPES = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"]

unreviewed = sorted(set(found_types) - set(REVIEWED_TRUSTED_TYPES))
if unreviewed:
    stop(f"model.skops contains types nobody has reviewed: {unreviewed}. Find out what each one is, and add it "
         "to REVIEWED_TRUSTED_TYPES only if you trust it.")
print("Every type in model.skops has been reviewed.")

Next, the rest of the bundle: the preprocessing code (a copy of the same file), the environment, the **schema** and the **model card**. Everything that can be generated from the run is generated, and only judgement calls need a human. That is the deck's *Documentation and the Hand-off to Governance* slide.

In [ ]:
def write_bundle(folder, model, test_acc, per_class, run_id, version_note):
    folder.mkdir(exist_ok=True)
    sio.dump(model, folder / "model.skops")
    shutil.copy(WORK / "preprocess.py", folder / "preprocess.py")
    (folder / "environment.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")
    schema = {
        "model_name": "mnist-digit-classifier",
        "owner": USER,
        "trained_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "note": version_note,
        "input": {
            "accepts": "an image of ONE handwritten digit: PNG/JPEG/WebP, any size, light-on-dark or dark-on-light",
            "model_input": {"dtype": "float32", "shape": [784], "range": [0.0, 1.0],
                            "produced_by": "preprocess.to_model_input",
                            "preprocess_version": pp.PREPROCESS_VERSION, "preprocess_sha256": PREPROCESS_SHA},
        },
        "output": {"classes": [int(c) for c in model.classes_],
                   "fields": ["prediction", "probabilities", "model_version", "request_id", "input_sha256"]},
        "metrics": {"test_accuracy": round(test_acc, 4), "per_class_test_accuracy": per_class,
                    "test_size": int(len(y_test))},
        "data": {"dataset": DATASET_ID, "revision": DATASET_REVISION, "split_hash": SPLIT_HASH,
                 "n_train": int(len(y_train)), "n_test": int(len(y_test))},
        "training": {"mlflow_run_id": run_id, "hyperparameters": BEST_PARAMS, "seed": SEED,
                     "environment_hash": ENV_HASH},
        "trusted_types": REVIEWED_TRUSTED_TYPES,
        "lab_run_id": RUN_ID,
    }
    (folder / "schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
    rows = "\n".join(f"| {c} | {a:.3f} |" for c, a in per_class.items())
    (folder / "README.md").write_text(f"""---
library_name: sklearn
tags:
- mlops-lab
- image-classification
datasets:
- {DATASET_ID}
metrics:
- accuracy
---

# MNIST digit classifier (MLOps lab run `{RUN_ID}`)

**Owner:** {USER}. **Status:** teaching artifact. It was created by the Tahakom MLOps lab and is deleted at the end of it. Not for any real use.
**This version:** {version_note}

## Intended use
Recognising one handwritten digit, 0-9, in a demo app.
**Out of scope:** any decision with consequences for a person; multi-digit strings; images that are not digits (the model has no "none of these" answer, so it always returns a digit).

## Training data
`{DATASET_ID}` at revision `{DATASET_REVISION}`: a stratified sample of {len(y_train)} images (split hash `{SPLIT_HASH}`), converted by `preprocess.py` (sha256 `{PREPROCESS_SHA}`, shipped in this same commit).

## Performance
Test accuracy **{test_acc:.4f}** on {len(y_test)} held-out images.

| class | test accuracy |
|---|---|
{rows}

## Known limitations
- The data is 1990s American handwriting. Other styles, and other numeral systems such as Eastern Arabic digits, are out of distribution.
- Rotated, blurred and low-contrast images degrade it; section 8 of the lab measures by how much.
- A high probability is not evidence of a correct answer.

## Lineage
MLflow run `{run_id}` (in the lab's local tracking store) → this commit. **The commit sha of this repository is the model version.** Every prediction the app serves carries it.

*Card format after Mitchell et al., "Model Cards for Model Reporting", FAT\\* 2019.*
""", encoding="utf-8")
    return schema

schema_v1 = write_bundle(BUNDLE_V1, model_v1, TEST_ACC, PER_CLASS, V1_RUN_ID,
                         version_note="v1, selected by the tpe+pruning study")
for f in sorted(BUNDLE_V1.iterdir()):
    print(f"  {f.name:<18} {f.stat().st_size / 1024:6.1f} KB")
print(json.dumps({k: schema_v1[k] for k in ("input", "output", "trusted_types")}, indent=1))

### The cheapest regression test in ML

Reload the artifact **from disk**, the way the server will:
- the pipeline through skops, with the reviewed trust list;
- `preprocess.py` from the bundle folder, not from this notebook's memory.

Then rebuild the test features from the **raw images** with that copy, and demand **bit-identical** probabilities on the whole test set.

It costs a second. It catches a whole class of packaging bugs: a step missing from the pipeline, a preprocessing file that differs from the one used in training, a library whose defaults changed between save and load.

In [ ]:
import importlib.util

def import_file(name, path):
    # Import a .py file by path, without leaving compiled bytecode (__pycache__) next to it.
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.dont_write_bytecode, before = True, sys.dont_write_bytecode
    try:
        spec.loader.exec_module(module)
    finally:
        sys.dont_write_bytecode = before
    return module

bundle_pp = import_file("bundle_preprocess", BUNDLE_V1 / "preprocess.py")
reloaded = sio.load(BUNDLE_V1 / "model.skops", trusted=REVIEWED_TRUSTED_TYPES)

X_test_again = bundle_pp.to_model_batch(test_images)     # rebuilt from raw images by the bundled code
assert np.array_equal(X_test_again, X_test), "the bundled preprocess.py does not reproduce the training features"
p_memory, p_disk = model_v1.predict_proba(X_test), reloaded.predict_proba(X_test_again)
assert np.array_equal(p_memory, p_disk), "the reloaded model disagrees with the in-memory model"
print(f"OK: {len(p_disk)} test images, probabilities bit-identical (max |difference| = {np.abs(p_memory - p_disk).max()})")

> **Exercise (TASK 1):** the deck's *Testing the Model* slide names **invariance tests**: "*perturbations that must **not** change the prediction*". The preprocessing contract promises two invariances. Inverting the colours, and moving the digit to a different place on a larger canvas, must not change the model input, and therefore must not change the prediction. Write the test. If you skip it, the rest of the lab still runs.

> **Try it** afterwards: replace the move with `ImageChops.offset(img, 3, 3)`. That shifts the 28×28 image *and wraps pixels around the edge*, and the share drops well below 100%. Is that a bug in the model, in the preprocessing, or in the test?

In [ ]:
from PIL import ImageChops, ImageOps

def check_invariances(images):
    # Return (share unchanged under colour inversion, share unchanged when the digit is moved).
    # TASK 1: build three batches with bundle_pp.to_model_batch: the originals; ImageOps.invert(img)
    #   for each; and each image moved on a bigger canvas:
    #       canvas = Image.new("L", (48, 48), 0); canvas.paste(img, (17, 11))
    #   Predict all three with `reloaded` and compare the predicted classes element-wise.
    # HINT: (pred_original == pred_inverted).mean() is the first number.
    raise NotImplementedError("TASK 1: implement check_invariances")

try:
    inverted_ok, moved_ok = check_invariances(test_images[:200])
    print(f"unchanged under inversion: {inverted_ok:.1%} | unchanged when moved: {moved_ok:.1%}")
except NotImplementedError as e:
    print(f"{e}: skipped. The rest of the lab does not depend on it.")

## 5. Register: versions, tags and an alias

> **From the deck, Versioning & Governance → *The Model Registry: A Lifecycle, Not a Folder*:** "*Deployments should reference a **mutable alias** ("champion", "challenger") that points at an **immutable version**. Promotion is then a metadata change, not a redeploy - and rollback is repointing the alias.*" And: "*Never delete the previous production version. It is your rollback target and your audit record.*"

The deck shows this with MLflow's registry. A Hugging Face model repository is a git repository, which gives us the same parts with nothing to install, plus a public URL:

| Deck (MLflow registry) | Here (Hugging Face model repo) |
|---|---|
| registered model | the repo `mlops-lab-<run id>-digits` |
| version *n* (immutable) | a **commit sha** on `main`, plus a git **tag** (`v1`, `v2`) for humans |
| alias `champion` | a **branch** named `champion`, pointing at one commit |
| Candidate | a commit no alias points at yet |
| Staging | the branch `challenger` (section 9) |
| Production | whatever `champion` points at |
| Archived | an older commit: still in the history, no longer served |
| promote / roll back | `delete_branch` + `create_branch(revision=<sha>)` |

(A git tag *can* be deleted and re-pointed, so treat tags as immutable by convention. The commit sha is the identity that truly cannot change.)

The upload names its files explicitly (`SHIP_ONLY`). A bundle folder collects junk, such as a `__pycache__` of compiled bytecode the moment anything imports from it, and whatever reaches the registry has been shipped.

In [ ]:
def require_hub(section):
    if api is None:
        stop(f"Section {section} needs a Hugging Face account, and you are in Plan B. See the appendix.")

def tag_version(tag, sha):
    # Give a commit a readable name. On a re-run of the lab the tag moves to the new commit, and we say so.
    tags = {t.name: t.target_commit for t in with_retry("Reading tags", lambda: api.list_repo_refs(MODEL_REPO_ID)).tags}
    if tags.get(tag) == sha:
        return
    if tag in tags:
        print(f"  (re-run) moving tag {tag} from {tags[tag][:10]} to {sha[:10]}")
        with_retry(f"Removing tag {tag}", lambda: api.delete_tag(MODEL_REPO_ID, tag=tag))
    with_retry(f"Tagging {tag}", lambda: api.create_tag(MODEL_REPO_ID, tag=tag, revision=sha))

ALIAS_HISTORY = []

def set_alias(alias, sha):
    # Promotion AND rollback are both this: point a branch at an immutable commit.
    branches = {b.name: b.target_commit for b in with_retry("Reading branches", lambda: api.list_repo_refs(MODEL_REPO_ID)).branches}
    old = branches.get(alias)
    if old != sha:
        if old is not None:
            with_retry(f"Removing the old {alias}", lambda: api.delete_branch(MODEL_REPO_ID, branch=alias))
        with_retry(f"Pointing {alias}", lambda: api.create_branch(MODEL_REPO_ID, branch=alias, revision=sha))
    resolved = with_retry("Checking the alias", lambda: api.model_info(MODEL_REPO_ID, revision=alias).sha)
    assert resolved == sha, f"{alias} resolves to {resolved}, expected {sha}"
    ALIAS_HISTORY.append({"at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                          "alias": alias, "from": old, "to": sha})
    print(f"  {alias}: {(old or 'none')[:10]} -> {sha[:10]}")

require_hub(5)
with_retry("Creating the model repo", lambda: api.create_repo(MODEL_REPO_ID, repo_type="model", private=False, exist_ok=True))
SHIP_ONLY = ["model.skops", "preprocess.py", "schema.json", "README.md", "environment.json"]   # nothing else
commit_v1 = with_retry("Uploading v1", lambda: api.upload_folder(
    repo_id=MODEL_REPO_ID, folder_path=BUNDLE_V1, allow_patterns=SHIP_ONLY,
    commit_message=f"v1: pca+scale+mlp {BEST_PARAMS}, MLflow run {V1_RUN_ID}"))
MODEL_V1 = commit_v1.oid          # the immutable version
tag_version("v1", MODEL_V1)
set_alias("champion", MODEL_V1)   # the mutable alias the deployment will follow
print(f"registered {MODEL_REPO_ID} @ {MODEL_V1}")
print(f"browse it: https://huggingface.co/{MODEL_REPO_ID}/tree/champion")

The registry now points back at the run: the commit message names the MLflow run id. Next, make the run point forward at the registry, so the chain can be walked in both directions.

> **From the deck, *Lineage: Answering "Where Did This Number Come From?"*:** "*raw source → dataset version → training run → model version → deployment → logged prediction*: *you must be able to walk this chain backwards*." Nothing exotic is needed, "*an ID at each hop, stored with the next hop*".

In [ ]:
require_hub(5)
tracker.set_tag(V1_RUN_ID, "registry.repo", MODEL_REPO_ID)
tracker.set_tag(V1_RUN_ID, "registry.version", MODEL_V1)
tracker.set_tag(V1_RUN_ID, "registry.tag", "v1")

def show_lineage(extra=()):
    hops = [
        ("raw source", f"{DATASET_ID} on the Hugging Face Hub"),
        ("dataset version", f"revision {DATASET_REVISION[:10]}, split {SPLIT_HASH}"),
        ("training run", f"MLflow run {V1_RUN_ID[:12]} (env {ENV_HASH}, preprocess {PREPROCESS_SHA})"),
        ("model version", f"{MODEL_REPO_ID} @ {MODEL_V1[:10]} (tag v1)"),
        ("alias", f"champion -> {MODEL_V1[:10]}"),
        *extra,
    ]
    for i, (hop, value) in enumerate(hops):
        print(f"  {hop:<18} {value}")
        if i < len(hops) - 1:
            print(f"  {'':<18} |")

show_lineage(extra=[("deployment", "(section 6)"), ("logged prediction", "(section 7)")])

## 6. Deploy: a live app with a public URL

> **From the deck, *Batch, Online, Streaming*:** "***Default to batch.** If a nightly table of predictions solves the problem, build that: an order of magnitude cheaper to run and to operate, and it fails without waking anyone.*"

This lab picks **online** anyway, so let us say out loud why: online is the pattern you can *see*. You can open it on your phone, draw a digit, and watch the model version come back in the response. A nightly batch job would teach the same registry and monitoring lessons, but its output is a table, and a table does not show you what "deployed" means. Exercise 2 at the end asks you to argue the cost difference.

**What gets deployed** follows the rules on the deck's *A Minimal Online Service* slide:
- **Load the model once at start-up**, from the registry, by alias. The alias is resolved to one immutable commit, and the model, its preprocessing code and its schema all come from that commit.
- **Return the model version with every prediction**: "*without it you cannot attribute a bad outcome to a model later*".
- **Load only the types written in `schema.json`** (section 4), and refuse anything else.
- **Log one JSON line per request**: the server's own record.
- **Watch the alias.** When `champion` moves, load the new version and swap it in. That is the deck's "*promotion is a metadata change*", implemented.

**Where it runs:** as a separate Python process in your Colab VM, configured only through environment variables. It is published through a **Gradio share link**: Gradio's share server forwards requests to your VM and stores nothing. The deck's *Containers, in One Slide* says "*do not bake giant model files into the image - pull them from the registry at start-up, by version*", and that is exactly what this process does.

> **What a real platform would add:** a stable URL instead of one that changes on every restart, a container image built from `requirements.txt` and referred to by digest, automatic restarts when the process crashes, health checks, and autoscaling. Hugging Face Spaces offered this for free until recently; the *What this costs* box at the top explains why this lab does not use them.

In [ ]:
%%writefile mlops_lab_work/serving/app.py
"""MNIST digit classifier - the serving process for the Tahakom MLOps lab.

Configuration comes ONLY from environment variables, never from the notebook's memory:
  MODEL_REPO          registry repo on the Hugging Face Hub (<namespace>/mlops-lab-...-digits)
  MODEL_REVISION      what to serve: an alias (branch) such as `champion`, or a commit sha
  ALIAS_POLL_SECONDS  how often to re-check where the alias points (0 = never)
  MODEL_DIR           Plan B only: serve a local bundle folder instead of the registry
  SHARE               "1" = ask Gradio for a public share URL
The process holds NO token: the model repo is public, and serving only needs to read it.
"""
import datetime as dt
import hashlib
import importlib.util
import json
import os
import threading
import time
import uuid
from pathlib import Path

import gradio as gr
import numpy as np
import skops.io as sio

APP_VERSION = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()[:12]   # this file's own digest


def log(event, **fields):
    """One JSON line per event on stdout: this is the server's log."""
    print(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="milliseconds"),
                      "event": event, **fields}), flush=True)


def load_bundle(get_file, meta):
    """Model + preprocessing code + schema, all from ONE version."""
    spec = importlib.util.spec_from_file_location("preprocess", get_file("preprocess.py"))
    preprocess = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(preprocess)   # code from the registry: only ever YOUR repo, pinned to one commit

    with open(get_file("schema.json"), encoding="utf-8") as f:
        schema = json.load(f)

    # Load only types a human reviewed and wrote into schema.json. Anything new: refuse.
    model_path = get_file("model.skops")
    found = sio.get_untrusted_types(file=model_path)
    unreviewed = sorted(set(found) - set(schema["trusted_types"]))
    if unreviewed:
        raise RuntimeError(f"refusing to load: unreviewed types in the model file: {unreviewed}")
    model = sio.load(model_path, trusted=found)
    return {"model": model, "preprocess": preprocess, "schema": schema, "meta": meta}


def load_from_hub(repo_id, revision):
    """Resolve the mutable alias to an immutable commit ONCE, then fetch every file at that commit."""
    from huggingface_hub import HfApi, hf_hub_download
    sha = HfApi().model_info(repo_id, revision=revision).sha
    meta = {"model_repo": repo_id, "model_alias": revision, "model_version": sha}
    return load_bundle(lambda name: hf_hub_download(repo_id, name, revision=sha), meta)


def load_from_dir(folder):
    """Plan B: a local bundle. The 'version' is a hash of the model file, since there is no registry."""
    folder = Path(folder)
    digest = hashlib.sha256((folder / "model.skops").read_bytes()).hexdigest()[:12]
    meta = {"model_repo": "local folder, no registry", "model_alias": None, "model_version": f"local-{digest}"}
    return load_bundle(lambda name: str(folder / name), meta)


class Server:
    """Holds the bundle being served. Swapping it is a single assignment, so every request
    is answered by one complete version, never half of the old one and half of the new."""

    def __init__(self, bundle):
        self.bundle = bundle

    def predict(self, image):
        b = self.bundle                        # read once: this request is served by this version
        t0 = time.perf_counter()
        if image is None:
            raise gr.Error("No image received.")
        x = b["preprocess"].to_model_input(image)
        proba = b["model"].predict_proba(x.reshape(1, -1))[0]
        classes = [int(c) for c in b["model"].classes_]
        record = {
            "request_id": uuid.uuid4().hex[:12],
            "served_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="milliseconds"),
            "prediction": classes[int(np.argmax(proba))],
            "probabilities": {str(c): float(p) for c, p in zip(classes, proba)},
            "input_sha256": b["preprocess"].feature_hash(x),
            "server_ms": round((time.perf_counter() - t0) * 1000, 3),
            **b["meta"],
            "app_version": APP_VERSION,
        }
        log("prediction", **{k: v for k, v in record.items() if k not in ("probabilities", "served_at")})
        return record

    def watch_alias(self, repo_id, alias, every):
        """Promotion and rollback are metadata changes: re-check where the alias points, and when
        it moves, load the new version and swap it in. No restart, no redeploy."""
        from huggingface_hub import HfApi
        api = HfApi()
        while True:
            time.sleep(every)
            try:
                sha = api.model_info(repo_id, revision=alias).sha
                if sha != self.bundle["meta"]["model_version"]:
                    old = self.bundle["meta"]["model_version"]
                    self.bundle = load_from_hub(repo_id, alias)
                    log("alias_moved", alias=alias, old=old, new=self.bundle["meta"]["model_version"])
            except Exception as e:                  # keep serving the version we have
                log("alias_check_failed", error=f"{type(e).__name__}: {e}"[:300])


def build_demo(server):
    meta, schema = server.bundle["meta"], server.bundle["schema"]
    with gr.Blocks(title="MNIST - MLOps lab") as demo:
        gr.Markdown(
            "## Handwritten digit classifier\n"
            f"Serving `{meta['model_repo']}`. Test accuracy at training time: "
            f"**{schema['metrics']['test_accuracy']:.3f}**. Draw one digit, large and centred. "
            "Every response names the exact model version that answered."
        )
        with gr.Tab("Draw"):
            sketch = gr.Sketchpad(label="Draw a digit", type="pil", image_mode="RGBA",
                                  canvas_size=(280, 280), format="png",
                                  brush=gr.Brush(default_size=18, colors=["#000000"], color_mode="fixed"))
            sketch_btn = gr.Button("Predict", variant="primary")
        with gr.Tab("Upload"):
            upload = gr.Image(label="Upload an image of one digit", type="pil", image_mode=None,
                              sources=["upload", "clipboard"], format="png")
            upload_btn = gr.Button("Predict", variant="primary")
        out = gr.JSON(label="Response")
        sketch_btn.click(server.predict, inputs=sketch, outputs=out, api_name="predict_sketch")
        upload_btn.click(server.predict, inputs=upload, outputs=out, api_name="predict")
    return demo


if __name__ == "__main__":
    t0 = time.perf_counter()
    if os.environ.get("MODEL_DIR"):
        bundle = load_from_dir(os.environ["MODEL_DIR"])
    else:
        bundle = load_from_hub(os.environ["MODEL_REPO"], os.environ.get("MODEL_REVISION", "champion"))
    server = Server(bundle)
    log("model_loaded", app_version=APP_VERSION, **bundle["meta"])

    every = float(os.environ.get("ALIAS_POLL_SECONDS", "0"))
    if every > 0 and not os.environ.get("MODEL_DIR"):
        threading.Thread(target=server.watch_alias, daemon=True,
                         args=(os.environ["MODEL_REPO"], os.environ.get("MODEL_REVISION", "champion"), every)).start()

    demo = build_demo(server).queue(default_concurrency_limit=8)
    _, local_url, public_url = demo.launch(share=os.environ.get("SHARE") == "1", prevent_thread_lock=True,
                                           quiet=True, ssr_mode=False, show_error=True)
    log("ready", local_url=local_url, public_url=public_url, startup_s=round(time.perf_counter() - t0, 2))
    demo.block_thread()

In [ ]:
# The serving environment, pinned to exactly what the model was trained with. In this lab the server runs
# in the same Colab VM, so this file documents the environment rather than installing it. On a real host
# (a container, a Space) it is what `pip install -r` reads, and an unpinned line here is how a model
# breaks six months later.
SERVING_LIBS = ["numpy", "scipy", "scikit-learn", "skops", "pillow", "huggingface_hub", "gradio"]
requirements = f"# python {ENV['python']}\n" + "\n".join(f"{lib}=={ENV[lib]}" for lib in SERVING_LIBS) + "\n"
(WORK / "serving" / "requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)

### Smoke test before you ship

> **From the deck, *Testing the Model*:** a **smoke test** asserts that "*the model loads, accepts one row, returns a probability in [0,1]. Catches most deployment breakage.*"

We import the server's own code and run its loader against the local bundle, here in the notebook, before anything goes on the internet.

In [ ]:
serving_app = import_file("serving_app", WORK / "serving" / "app.py")
smoke = serving_app.Server(serving_app.load_from_dir(BUNDLE_V1))
response = smoke.predict(test_images[0])

p = np.array(list(response["probabilities"].values()))
assert response["prediction"] in range(10), "prediction is not a digit"
assert np.all((p >= 0) & (p <= 1)) and abs(p.sum() - 1) < 1e-6, "probabilities are not a distribution"
assert response["input_sha256"] == pp.feature_hash(X_test[0]), "the server's preprocessing differs from training"
assert response["model_version"] and response["app_version"], "the response does not say which model answered"
print("smoke test passed:")
print(json.dumps(response, indent=1))

### Start the server

`start_server` launches `app.py` as its own process, with a minimal environment:
- the registry coordinates, passed as **configuration** (`MODEL_REPO`, `MODEL_REVISION`, `ALIAS_POLL_SECONDS`);
- **no token**.

Those values are not secret: anyone can see a public repo, and keeping them as plain configuration documents that. A **secret** is anything whose leak would hurt, such as your token. A secret belongs in a secret store, never in plain configuration or a log. And because this server only ever *reads* a public repo, it gets no credentials at all. That is **least privilege**.

The server writes its log to `mlops_lab_work/serving/server.log`: one JSON line per event.

In [ ]:
SERVER = {"proc": None}
SERVER_LOG = WORK / "serving" / "server.log"
PID_FILE = WORK / "serving" / "server.pid"

def server_events():
    # The server's log as a list of dicts. Non-JSON lines (Gradio's own messages) are skipped.
    events = []
    if SERVER_LOG.exists():
        for line in SERVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.startswith("{"):
                try:
                    events.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return events

def _is_our_server(pid):
    # Only ever signal a process that is running our app.py (Linux/Colab can check; elsewhere, don't guess).
    try:
        return b"app.py" in Path(f"/proc/{pid}/cmdline").read_bytes()
    except OSError:
        return False

def stop_server():
    proc = SERVER.get("proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            proc.kill()
    elif PID_FILE.exists():              # a server left over from an earlier kernel of this notebook
        pid = int(PID_FILE.read_text())
        if _is_our_server(pid):
            os.kill(pid, signal.SIGTERM)
    PID_FILE.unlink(missing_ok=True)
    SERVER["proc"] = None

def start_server(config, timeout=180):
    stop_server()
    env = {k: v for k, v in os.environ.items() if not k.startswith("HF_TOKEN")}   # no credentials for the server
    env.update({"HF_HUB_DISABLE_IMPLICIT_TOKEN": "1", "GRADIO_ANALYTICS_ENABLED": "False",
                "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", **config})
    t0 = time.perf_counter()
    with open(SERVER_LOG, "w", encoding="utf-8") as log_file:
        proc = subprocess.Popen([sys.executable, "app.py"], cwd=WORK / "serving", env=env,
                                stdout=log_file, stderr=subprocess.STDOUT)
    SERVER["proc"] = proc
    PID_FILE.write_text(str(proc.pid))
    while time.perf_counter() - t0 < timeout:
        ready = [e for e in server_events() if e.get("event") == "ready"]
        if ready:
            return ready[-1]
        if proc.poll() is not None:
            break
        time.sleep(1)
    print("---- last lines of the server log ----")
    print("\n".join(SERVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-25:]))
    stop(("The server exited during start-up" if proc.poll() is not None else f"The server was not ready after {timeout}s")
         + ". The log above says why. The troubleshooting notes below this cell list the usual causes.")

if PLAN_B:
    CONFIG = {"MODEL_DIR": str(BUNDLE_V1.resolve()), "SHARE": "1"}          # see the appendix
else:
    CONFIG = {"MODEL_REPO": MODEL_REPO_ID, "MODEL_REVISION": "champion", "ALIAS_POLL_SECONDS": "10", "SHARE": "1"}

READY = start_server(CONFIG)
LOCAL_URL = READY["local_url"]
PUBLIC_URL = READY["public_url"] or LOCAL_URL
LOADED = next(e for e in server_events() if e["event"] == "model_loaded")
SERVED_VERSION = LOADED["model_version"]
if not PLAN_B:
    assert SERVED_VERSION == MODEL_V1, f"the server loaded {SERVED_VERSION}, but champion is {MODEL_V1}"

print(f"server ready in {READY['startup_s']}s (pid {SERVER['proc'].pid})")
print(f"serving   : {LOADED['model_repo']} @ {SERVED_VERSION[:10]}  (alias: {LOADED['model_alias']})")
print(f"app       : app.py digest {LOADED['app_version']}")
print(f"local URL : {LOCAL_URL}")
if not READY["public_url"]:
    print("WARNING: no public URL. The share tunnel could not be created, but everything below still works "
          "against the local URL.")
display(Markdown(f"## Your model is live: [{PUBLIC_URL}]({PUBLIC_URL})\n"
                 "Open it on your phone (send yourself the link), draw a digit, and find `model_version` in the response."))

> **If the server does not come up,** the cell prints the tail of `mlops_lab_work/serving/server.log`:
> - `RepositoryNotFoundError` or `RevisionNotFoundError`: section 5 did not finish, so the repo or the `champion` branch does not exist yet. Re-run section 5.
> - `refusing to load: unreviewed types`: the model file contains a type that is not in `schema.json`'s `trusted_types`. This is working as intended; review the type (section 4).
> - The server is ready but there is no public URL: the share tunnel was blocked. This is rare on Colab and common on corporate networks. Sections 7-8 still work against the local URL.
> - The public link stops answering later: Colab disconnected, or the kernel restarted, and the tunnel dies with the process. That is the price of not having a platform. Re-run this cell and you get a new URL.
>
> `server_events()[-5:]` shows the most recent entries of the server log at any time.

## 7. Inference over the wire

The model now runs in a separate process and is published on the internet. From here on, the notebook talks to it the way any client would: over HTTP. `model_v1` in this notebook is used only to *compare* against what comes back.

In [ ]:
from gradio_client import Client, handle_file
from concurrent.futures import ThreadPoolExecutor

def connect(url):
    return with_retry(f"Connecting to {url}", lambda: Client(url, verbose=False, httpx_kwargs={"timeout": 30}),
                      hint="that the server cell in section 6 is still running (Colab may have disconnected)")

def save_png(img, path):
    img.save(path, format="PNG")
    return path

def call(client, png_path, attempts=3):
    # One prediction over HTTP. Returns (response, round-trip seconds). Retries transient failures.
    for attempt in range(1, attempts + 1):
        t0 = time.perf_counter()
        try:
            response = client.submit(handle_file(str(png_path)), api_name="/predict").result(timeout=60)
            return response, time.perf_counter() - t0
        except Exception as e:
            if attempt == attempts:
                stop(f"A prediction call failed {attempts} times ({type(e).__name__}: {str(e)[:200]}). "
                     "Check that the server is still running: server_events()[-5:]")
            time.sleep(2 * attempt)

t0 = time.perf_counter()
client = connect(PUBLIC_URL)
connect_s = time.perf_counter() - t0
first_png = save_png(test_images[0], WORK / "traffic" / "first.png")
first, first_s = call(client, first_png)

print(f"connect (download the API description) : {connect_s * 1000:6.0f} ms")
print(f"first prediction, cold                   : {first_s * 1000:6.0f} ms, of which inside the model: {first['server_ms']} ms")
print(json.dumps(first, indent=1))

### It is just HTTP

`gradio_client` is a convenience. Underneath, one prediction is three plain HTTP requests:
1. upload the image file;
2. start a call that refers to it;
3. read the result from a server-sent-event stream.

Here they are with `requests`, so you could do the same from `curl`, JavaScript or a phone app.

In [ ]:
def predict_raw(base_url, png_path, api_name="predict", timeout=30):
    base = base_url.rstrip("/")
    with open(png_path, "rb") as f:                                                      # 1. upload
        up = requests.post(f"{base}/gradio_api/upload",
                           files={"files": (Path(png_path).name, f, "image/png")}, timeout=timeout)
    up.raise_for_status()
    file_ref = {"path": up.json()[0], "meta": {"_type": "gradio.FileData"}}
    started = requests.post(f"{base}/gradio_api/call/{api_name}", json={"data": [file_ref]}, timeout=timeout)  # 2. call
    started.raise_for_status()
    stream = requests.get(f"{base}/gradio_api/call/{api_name}/{started.json()['event_id']}", timeout=timeout)  # 3. result
    stream.raise_for_status()
    print("raw event stream:", stream.text[:160].replace("\n", " | "), "...")
    for block in stream.text.split("\n\n"):
        lines = block.strip().splitlines()
        if lines and lines[0] == "event: complete":
            return json.loads(lines[1].removeprefix("data: "))[0]
        if lines and lines[0] == "event: error":
            raise RuntimeError(block)
    raise RuntimeError("the stream ended without a 'complete' event")

raw = with_retry("A raw HTTP prediction", lambda: predict_raw(PUBLIC_URL, first_png))
print(f"\nprediction {raw['prediction']} from model_version {raw['model_version'][:10]}, request {raw['request_id']}")

### Parity: does the server compute what we computed?

> **From the deck:** "*one artifact, one `predict`, no room for skew*." This is the test that *proves* there is no skew, instead of asserting it.

We send **400 randomly chosen held-out test images** to the server:
- the first **100 through the public URL**, the full internet path;
- the other **300 through the server's local address**.

Both halves go through exactly the same server code: each image is PNG-encoded, uploaded, decoded by Gradio, preprocessed by the server's own copy of `preprocess.py`, and scored by the model the server downloaded from the registry. The split only spares Gradio's relay, a free service shared by everyone in the room: in our tests, parallel requests through it barely overlapped, so bulk traffic over it is slow. Then we compare with what the notebook computes locally for the same images.

Two things are compared separately, because they point at different bugs:
- **`input_sha256`**, the hash of the 784 numbers *as the model saw them*. A mismatch here is **preprocessing skew**: a different image decode, a different `preprocess.py`, a different Pillow.
- **The prediction and probabilities, given identical inputs.** A mismatch here is **model or environment skew**: a different model file, or a different scikit-learn or numpy.

One subtlety the output shows: we score the local copy **one row at a time**, the way the server does. Score the same 400 images as a single batch and the probabilities differ from the served ones around the sixth decimal, with the same model on the same machine. In float32, a batched matrix product adds its terms in a different order than 400 single-row products, and floating-point addition is not associative: the deck's *Five Places Nondeterminism Hides*, live. So a parity test compares like with like: the predicted class must match exactly, and the probabilities are compared with a stated tolerance.

These 400 responses also become the **reference window** for monitoring in section 8: clean data, scored by the live endpoint. They are a *random* sample on purpose. MNIST's test file is not shuffled, because its halves come from different groups of writers, so "the first 400" would be a biased reference.

In [ ]:
PRED_LOG, LABEL_LOG = WORK / "predictions.jsonl", WORK / "labels.jsonl"
WINDOW_RECORDS = {}

def log_predictions(results, labels, window):
    # Client-side prediction log: one JSON line per request. Labels go to a separate file and are joined
    # later on request_id, because in production they arrive days or months after the prediction.
    records = []
    with open(PRED_LOG, "a", encoding="utf-8") as preds, open(LABEL_LOG, "a", encoding="utf-8") as labs:
        for (response, seconds), label in zip(results, labels):
            rec = {"ts": response["served_at"], "window": window, "request_id": response["request_id"],
                   "input_sha256": response["input_sha256"], "prediction": response["prediction"],
                   "probabilities": response["probabilities"], "model_version": response["model_version"],
                   "latency_ms": round(seconds * 1000, 1), "server_ms": response["server_ms"]}
            preds.write(json.dumps(rec) + "\n")
            labs.write(json.dumps({"request_id": rec["request_id"], "label": int(label)}) + "\n")
            records.append(rec)
    WINDOW_RECORDS.setdefault(window, []).extend(records)
    return records

def send(client, images, labels, window, workers=8):
    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", window)
    paths = [save_png(img, WORK / "traffic" / f"{safe}-{i:04d}.png") for i, img in enumerate(images)]
    with ThreadPoolExecutor(workers) as pool:
        results = list(pool.map(lambda path: call(client, path), paths))
    return log_predictions(results, labels, window)

for f in (PRED_LOG, LABEL_LOG):
    f.unlink(missing_ok=True)                 # one fresh log per run of this section
WINDOW_RECORDS.clear()

N_REF, N_PUBLIC = 400, 100
rng = np.random.default_rng(SEED)
positions = rng.permutation(len(test_idx))
ref_pos, probe_pos = np.sort(positions[:N_REF]), np.sort(positions[N_REF:N_REF + 200])   # probe set: section 9

ref_images, ref_labels = [test_images[i] for i in ref_pos], y_test[ref_pos]
t0 = time.perf_counter()
over_internet = send(client, ref_images[:N_PUBLIC], ref_labels[:N_PUBLIC], window="reference")
print(f"{N_PUBLIC} calls through the public URL in {time.perf_counter() - t0:.1f}s")

TRAFFIC_URL = LOCAL_URL               # bulk traffic: the same server process, without the shared relay
traffic_client = connect(TRAFFIC_URL)
t0 = time.perf_counter()
over_local = send(traffic_client, ref_images[N_PUBLIC:], ref_labels[N_PUBLIC:], window="reference")
print(f"{N_REF - N_PUBLIC} calls through the local address in {time.perf_counter() - t0:.1f}s")
ref_records = over_internet + over_local

served_proba = np.array([[r["probabilities"][str(c)] for c in CLASSES] for r in ref_records])
local_proba = np.array([model_v1.predict_proba(X_test[i:i + 1])[0] for i in ref_pos])   # one row at a time, like the server
batch_proba = model_v1.predict_proba(X_test[ref_pos])                                   # all 400 in one call
hash_mismatch = [int(i) for i, r in zip(ref_pos, ref_records) if r["input_sha256"] != pp.feature_hash(X_test[i])]
pred_mismatch = [int(i) for i, r, p in zip(ref_pos, ref_records, local_proba) if r["prediction"] != int(p.argmax())]
max_diff = float(np.abs(local_proba - served_proba).max())
print(f"input-hash mismatches : {len(hash_mismatch)}")
print(f"prediction mismatches : {len(pred_mismatch)}")
print(f"max |p_local - p_served|, local scored row by row : {max_diff:.2e}")
print(f"max |p_local - p_served|, local scored as a batch : {np.abs(batch_proba - served_proba).max():.2e}")

assert not hash_mismatch, (f"PREPROCESSING SKEW on test images {hash_mismatch[:10]}: the server turned the same image "
                           "into different numbers. Compare Pillow versions and the preprocess.py in the registry.")
assert not pred_mismatch and max_diff < 1e-6, (f"MODEL/ENVIRONMENT SKEW: same inputs, different outputs "
                                               f"(max diff {max_diff:.2e}). Compare scikit-learn/numpy versions and the model file.")
print("PARITY OK: the deployed service computes what the notebook computes.")

### Latency: where does the time go?

> **From the deck, *Protocols and Latency Budgets*:** "***Quote p95 and p99, never the mean.** The tail is what users experience and what times out upstream.*" And: "*The model is often the **smallest** term. Optimise what you measured, not what you assumed.*"

Every response carries `server_ms`: the time spent *inside* the prediction function, preprocessing plus model. The client's round trip adds everything else. We make 50 sequential calls (no concurrency, so no queueing) two ways:
- through the **public URL**: the internet, the tunnel, Gradio, and the model;
- through the **local URL**, from inside the VM: Gradio's own HTTP work and the model, but no internet.

Subtracting one from the other splits the budget into three terms.

In [ ]:
def latency_profile(url, png_path, n=50):
    c = connect(url)
    round_trip, inside = [], []
    for _ in range(n):
        response, seconds = call(c, png_path)
        round_trip.append(seconds * 1000)
        inside.append(response["server_ms"])
    return np.array(round_trip), np.array(inside)

public_rt, public_in = latency_profile(PUBLIC_URL, first_png)
local_rt, local_in = latency_profile(LOCAL_URL, first_png)

def pct(a):
    return {f"p{q}": round(float(np.percentile(a, q)), 1) for q in (50, 95, 99)}

latency = pd.DataFrame({"public URL, round trip (ms)": pct(public_rt),
                        "local URL, round trip (ms)": pct(local_rt),
                        "inside the model, server_ms": pct(np.concatenate([public_in, local_in]))}).T
display(latency)

model_ms = float(np.median(np.concatenate([public_in, local_in])))
framework_ms = float(np.median(local_rt)) - model_ms
network_ms = float(np.median(public_rt) - np.median(local_rt))
print(f"median budget: model {model_ms:.2f} ms | Gradio HTTP work (upload, queue, stream) {framework_ms:.0f} ms | "
      f"internet + tunnel {network_ms:.0f} ms")
print(f"the model is {100 * model_ms / np.median(public_rt):.2f}% of what a user waits for")
print(f"the cold first call took {first_s * 1000:.0f} ms, against a warm median of {np.median(public_rt):.0f} ms")

Compare this with the deck's budget table: network in/out 5-20 ms, preprocessing 1-5 ms, model forward pass 1-10 ms.
- **Your model term** is probably under a millisecond.
- **Your network term** is probably far above 20 ms, because every request here crosses a public tunnel, and Gradio spends three HTTP requests per prediction.

Neither number is wrong: the deck's table describes a service and its client in the same data centre. The lesson holds even more strongly here. If this endpoint were too slow, making the model faster would change nothing a user could feel.

> **Honest statistics:** a p99 from 50 samples is essentially the single slowest call. To quote a p99 you trust, you need thousands of requests.

### Every response names its model

> **From the deck, *Log Every Prediction You Serve*:** record "*the **model version** and config version*" with each request. Without it, the lineage slide's question, "*which model produced **this** decision?*", has no answer. A response without a version is unauditable: when a customer complains about a decision, you cannot tell whether the model that made it is the one running today.

In [ ]:
log_df = pd.read_json(PRED_LOG, lines=True)
print("model versions seen in the prediction log:")
print(log_df["model_version"].value_counts().to_string())
assert set(log_df["model_version"]) == {SERVED_VERSION}

request = log_df.iloc[-1]["request_id"]
print(f"\nthe server's own record of request {request}:")
print([e for e in server_events() if e.get("request_id") == request])

if not PLAN_B:
    print("\nThe whole chain, walkable backwards from one logged prediction:")
    show_lineage(extra=[("deployment", f"app.py {LOADED['app_version']}, a process on this VM, at {PUBLIC_URL}"),
                        ("logged prediction", f"request {request} -> model_version {SERVED_VERSION[:10]}")])

## 8. Monitoring and drift

> **From the deck, *Four Ways a Deployed Model Goes Wrong*:** "***Server uptime tells you none of this.** A model service returning 200s at 8 ms with catastrophically wrong predictions is a perfectly healthy service.*"

> **And from *Detecting Drift Before the Labels Arrive*:** watch the **prediction distribution** ("*if your model predicted 'churn' for 4% of users all year and today it says 19%, something changed*") and the **confidence** of predictions. Use **PSI** for categorical quantities and **Kolmogorov-Smirnov** for continuous ones. For PSI, the credit-scoring rule of thumb is "*< 0.1 stable, 0.1-0.25 moderate shift, > 0.25 significant. A convention, not a theorem - calibrate it on your own history.*"

Our two monitors, both computable the moment a prediction is made:
1. **PSI of the predicted-class distribution** (10 bins, one per class) against the reference window.
2. **KS statistic *D* on the confidence** (the top probability per request) against the reference window. We alert on the **effect size** *D*, not on the p-value, which is the deck's "*large-n trap*": with enough traffic, every test rejects.

A window **breaches** if PSI > 0.25 or *D* > 0.20. An **alert** fires only when **3 windows in a row** breach. That is the deck's *Alerts People Act On*: "*alert on a **sustained** threshold crossing, not on one noisy window.*"

First, check the maths on synthetic data where we know the answer.

In [ ]:
from scipy.stats import ks_2samp

PSI_MODERATE, PSI_SIGNIFICANT = 0.10, 0.25    # the credit-scoring convention quoted in the deck
KS_EFFECT = 0.20                              # our own line for KS: an effect size, not a p-value
SUSTAIN = 3                                   # alert only after this many breaching windows in a row

def psi(reference, current, bins=range(10), eps=1e-4):
    # Population Stability Index between two samples of a discrete variable (here: the predicted class).
    p = np.array([np.mean(np.asarray(reference) == b) for b in bins])
    q = np.array([np.mean(np.asarray(current) == b) for b in bins])
    p, q = np.clip(p, eps, None), np.clip(q, eps, None)    # an empty bin would make ln(p/q) infinite
    return float(np.sum((p - q) * np.log(p / q)))

def sustained(breaches, n=SUSTAIN):
    # True at window i if windows i-n+1 .. i all breached.
    return [i >= n - 1 and all(breaches[i - n + 1:i + 1]) for i in range(len(breaches))]

synthetic = np.random.default_rng(SEED)
uniform = synthetic.integers(0, 10, 5000)
print(f"PSI, same distribution            : {psi(uniform, synthetic.integers(0, 10, 5000)):.4f}   (expect ~0)")
print(f"PSI, class 0 grows from 10% to 30%: {psi(uniform, synthetic.choice(10, 5000, p=[0.3] + [0.7 / 9] * 9)):.4f}   (expect > 0.1)")
print(f"KS D, same confidence distribution: {ks_2samp(synthetic.beta(8, 1, 2000), synthetic.beta(8, 1, 2000)).statistic:.3f}")
print(f"KS D, confidence drops            : {ks_2samp(synthetic.beta(8, 1, 2000), synthetic.beta(4, 1, 2000)).statistic:.3f}")
print("sustained([F, T, F, T, T, T, T])  :", sustained([False, True, False, True, True, True, True]))

### How big is "nothing happened"?

A threshold only means something relative to the noise of *your* windows. Before looking at real traffic, measure that noise. We score 300 **clean** windows of the size we are about to use (200 requests), locally, from the traffic pool, and see how large PSI and *D* get when nothing has changed. This is the deck's "*calibrate it on your own history*", done before the history exists. It takes a few seconds and needs no network.

In [ ]:
WINDOW = 200
pool_images = list(ds["test"].select(pool_idx)["image"])
pool_labels = y_test_all[pool_idx]
P_pool = model_v1.predict_proba(pp.to_model_batch(pool_images))          # local only, for calibration
P_ref = model_v1.predict_proba(X_test[ref_pos])                            # identical to the served reference (parity)

floor = np.random.default_rng(SEED + 1)
noise = []
for _ in range(300):
    window = floor.choice(len(P_pool), WINDOW, replace=False)
    noise.append((psi(P_ref.argmax(1), P_pool[window].argmax(1)), ks_2samp(P_ref.max(1), P_pool[window].max(1)).statistic))
noise = np.array(noise)
print(f"clean windows of {WINDOW}: PSI  median {np.median(noise[:, 0]):.3f}, 95th pct {np.percentile(noise[:, 0], 95):.3f}, max {noise[:, 0].max():.3f}")
print(f"                      KS D median {np.median(noise[:, 1]):.3f}, 95th pct {np.percentile(noise[:, 1], 95):.3f}, max {noise[:, 1].max():.3f}")
print(f"clean windows over the PSI 'moderate' line ({PSI_MODERATE}): {np.mean(noise[:, 0] > PSI_MODERATE):.0%} - "
      "which is why that line alone would page someone for nothing")

### Two weeks of traffic, through the live endpoint

Each "day" sends one window of 200 images from the traffic pool, some of them altered, to the **running server**. As in section 7, the bulk traffic goes to the server's local address (`TRAFFIC_URL`). That way thirty students don't push 3,000 requests each through Gradio's free relay, and a flaky tunnel can't break the experiment. It is the same process, the same code and the same model. Set `TRAFFIC_URL = PUBLIC_URL` before this cell to route it over the internet instead, and expect it to take much longer.

| day | traffic |
|---|---|
| 01-02 | clean |
| 03 | colours inverted (dark ink on white) |
| 04 | one bad day: a smudged lens (blur, washed-out grey background) |
| 05 | clean |
| 06-09 | the scanner slowly tilts: rotated 15°, 25°, 35°, 45° |
| 10 | clean |
| 11-13 | a new customer sends mostly 1s (half of all images) |
| 14 | every 6 and 9 arrives upside down |

In [ ]:
from PIL import ImageFilter

TIMELINE = [
    ("day 01", "clean", None), ("day 02", "clean", None),
    ("day 03", "inverted", None),
    ("day 04", "smudged", 1.0),
    ("day 05", "clean", None),
    ("day 06", "rotated", 15), ("day 07", "rotated", 25), ("day 08", "rotated", 35), ("day 09", "rotated", 45),
    ("day 10", "clean", None),
    ("day 11", "prior shift", 1), ("day 12", "prior shift", 1), ("day 13", "prior shift", 1),
    ("day 14", "upside-down 6s and 9s", None),
]

def make_window(kind, param, rng):
    if kind == "prior shift":         # half the window is digit `param`, the rest is the usual mix
        is_param = pool_labels == param
        weights = np.where(is_param, 0.5 / is_param.sum(), 0.5 / (~is_param).sum())
        picks = rng.choice(len(pool_labels), WINDOW, replace=False, p=weights)
    else:
        picks = rng.choice(len(pool_labels), WINDOW, replace=False)
    images, labels = [], []
    for i in picks:
        img, label = pool_images[int(i)], int(pool_labels[i])
        if kind == "inverted":
            img = ImageOps.invert(img)
        elif kind == "smudged":
            blurred = np.asarray(img.filter(ImageFilter.GaussianBlur(param)), dtype=np.float32)
            img = Image.fromarray((blurred * 0.5 + 100).astype(np.uint8))
        elif kind == "rotated":
            img = img.rotate(param, resample=Image.Resampling.BILINEAR, fillcolor=0)
        elif kind == "upside-down 6s and 9s" and label in (6, 9):
            img = img.rotate(180)     # a rotated 6 looks like a 9: same pixels, different right answer
        images.append(img)
        labels.append(label)
    return images, labels

ref_pred = [r["prediction"] for r in ref_records]
ref_conf = [max(r["probabilities"].values()) for r in ref_records]
traffic_client = connect(TRAFFIC_URL)          # TRAFFIC_URL was set in section 7
traffic_rng = np.random.default_rng(SEED + 2)
rows = []
t0 = time.perf_counter()
for day, kind, param in TIMELINE:
    images, labels = make_window(kind, param, traffic_rng)
    records = send(traffic_client, images, labels, window=day)
    pred = [r["prediction"] for r in records]
    conf = [max(r["probabilities"].values()) for r in records]
    ks = ks_2samp(ref_conf, conf)
    rows.append({"window": day, "traffic": kind + (f" {param}" if kind == "rotated" else ""),
                 "psi_class": psi(ref_pred, pred), "ks_D_confidence": ks.statistic, "ks_p": ks.pvalue})
    print(f"{day}  {rows[-1]['traffic']:<24} PSI {rows[-1]['psi_class']:6.3f}   KS D {ks.statistic:5.3f}")
print(f"{len(TIMELINE) * WINDOW} requests in {time.perf_counter() - t0:.0f}s")

monitor = pd.DataFrame(rows)
monitor["psi_band"] = pd.cut(monitor["psi_class"], [-np.inf, PSI_MODERATE, PSI_SIGNIFICANT, np.inf],
                             labels=["stable", "moderate", "significant"])
monitor["breach"] = (monitor["psi_class"] > PSI_SIGNIFICANT) | (monitor["ks_D_confidence"] > KS_EFFECT)
monitor["ALERT"] = sustained(monitor["breach"].tolist())
monitor.round(3)

### Weeks later: the labels arrive

Everything above used only what exists at prediction time. Now suppose the true labels arrive. In production that takes days for fraud and months for credit default: the deck's *label-lag* slide. They are **joined to the prediction log on `request_id`**, and only now can we compute accuracy per window.

In [ ]:
joined = pd.read_json(PRED_LOG, lines=True).merge(pd.read_json(LABEL_LOG, lines=True), on="request_id", how="left")
accuracy = (joined["prediction"] == joined["label"]).groupby(joined["window"]).mean()
monitor["accuracy (known later)"] = monitor["window"].map(accuracy)
print(f"reference window accuracy: {accuracy['reference']:.3f}")
monitor.round(3)

In [ ]:
x = np.arange(len(monitor))
fig, axes = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True)

panels = [
    ("psi_class", "PSI, predicted class", [(PSI_MODERATE, "0.10 moderate"), (PSI_SIGNIFICANT, "0.25 significant")]),
    ("ks_D_confidence", "KS D, confidence", [(KS_EFFECT, "0.20 our line")]),
    ("accuracy (known later)", "accuracy (known only later)", []),
]
PSI_TOP = 1.2                                          # the smudged day is far above this; it is labelled
for ax, (column, title, lines) in zip(axes, panels):
    values = monitor[column].to_numpy(dtype=float)
    shown = np.minimum(values, PSI_TOP) if column == "psi_class" else values
    ax.plot(x, shown, "-o", color=BLUE, linewidth=2, markersize=7, markeredgecolor="white", markeredgewidth=2)
    if column == "psi_class":
        ax.set_ylim(0, PSI_TOP * 1.1)
        for i in np.flatnonzero(values > PSI_TOP):
            ax.annotate(f"{values[i]:.1f}, off the scale", (x[i], PSI_TOP), xytext=(8, -2),
                        textcoords="offset points", color=INK_2, fontsize=8, va="top")
    for level, label in lines:
        ax.axhline(level, color=INK_2, linestyle=":", linewidth=1)
        ax.text(-0.45, level, label, color=INK_2, fontsize=8, va="bottom", ha="left")
    ax.set_title(title, color=INK, fontsize=10, loc="left")
    tidy(ax)
for i in np.flatnonzero(monitor["ALERT"]):
    for ax in axes:
        ax.axvspan(i - 0.4, i + 0.4, color=CRITICAL, alpha=0.08, linewidth=0)
    axes[0].text(i, axes[0].get_ylim()[1], "ALERT", color=CRITICAL, fontsize=8, ha="center", va="top", fontweight="bold")
axes[-1].set_xticks(x, [f"{w}\n{t}" for w, t in zip(monitor["window"], monitor["traffic"])], rotation=60, ha="right", fontsize=8)
fig.suptitle("Two weeks of traffic: what the monitors saw, and what the labels said later", x=0.01, ha="left", color=INK)
plt.tight_layout()
plt.show()

> **Exercise (TASK 2):** the deck lists a third signal you can compute before the labels arrive: "*confidence / entropy of predictions*". Add a column `low_conf_rate`: the fraction of requests in each window whose top probability is below 0.6. Flag the windows where it is more than twice the reference window's rate. Which days does it catch? Does it catch day 14?

In [ ]:
def low_confidence_rate(records, threshold=0.6):
    # TASK 2: return the fraction of `records` whose top probability is below `threshold`.
    # HINT: max(r["probabilities"].values()) is the top probability of one record.
    raise NotImplementedError("TASK 2: implement low_confidence_rate")

try:
    reference_rate = low_confidence_rate(WINDOW_RECORDS["reference"])
    monitor["low_conf_rate"] = [low_confidence_rate(WINDOW_RECORDS[w]) for w in monitor["window"]]
    monitor["low_conf_flag"] = monitor["low_conf_rate"] > 2 * reference_rate
    print(f"reference low-confidence rate: {reference_rate:.3f}")
    display(monitor[["window", "traffic", "low_conf_rate", "low_conf_flag", "breach", "ALERT"]].round(3))
except NotImplementedError as e:
    print(f"{e}: skipped. The rest of the lab does not depend on it.")

### Which kind of drift was each one?

Answer these before opening the answers. The deck's *Which Kind Is It? Worked Examples* and *Drift Comes in Shapes* slides are the answer key.

1. **Inverted colours (day 03).** Every raw pixel changed, yet neither monitor moved. Why? Is that good?
2. **The smudged day (day 04).** It breached massively, but no alert fired. Should one have?
3. **Rotation (days 06-09).** Covariate drift or concept drift? When did the alert fire, and why not on day 06?
4. **The new customer (days 11-13).** The alert fired. Is the model wrong? Look at the accuracy column.
5. **Upside-down 6s and 9s (day 14).** What did PSI and KS say? What did accuracy say, weeks later?

Then: **what would you actually retrain on?**

<details>
<summary><b>Answers</b></summary>

1. **Covariate shift, absorbed by the preprocessing contract.** The raw input distribution moved completely, but `preprocess.py` flips light-on-dark images, so the model's *input* never changed. That is why the deck says to log and monitor the features "*as the model saw them*": a monitor on raw pixels would have paged someone for nothing. Nothing to retrain.
2. **An outlier, not drift.** The deck: "*A single anomalous window is an outlier, not drift. Do not retrain on one bad Tuesday.*" It also says "*always check for upstream breakage first*": a smudged lens is a hardware or data-engineering fix, and retraining on it would bake the breakage into the model. The sustained-crossing rule kept this from paging anyone. The drop in accuracy was real, so someone should look at it, in the weekly digest ("*page on breakage; e-mail a digest for drift*").
3. **Covariate shift, and incremental**: $P(X)$ moved while the rule stayed put, because a tilted 7 is still a 7. The deck calls incremental drift "*the dangerous ones. Any single week looks fine*", and it is caught only against a *fixed* reference window, which is what we used. The alert fires on the third breaching day in a row. The response is to fix the scanner if the tilt is a fault, or, if tilted input is the new normal, to retrain with rotated examples (augmentation) and gate the retrain.
4. **Label (prior) shift**: $P(Y)$ moved, but each class still looks the same. The prediction-distribution alert fired because the true mix really did change, yet accuracy is fine, since 1s are easy. Retraining buys nothing. The alert was still useful: it is the deck's churn example ("*4% all year and today it says 19%*"). Something changed in the world, and a human decides whether it matters. Dataset-shift literature treats this as its own kind (Quiñonero-Candela et al., in the deck's references).
5. **Concept drift**: $P(Y \mid X)$ moved. The pixels look like ordinary MNIST digits, and the class mix even stays balanced, but the right answer for "a 9-shaped image" is now sometimes 6. The deck: "*the inputs look identical and the answer changed. Nothing in the input distribution will tell you. Only labels will.*" Neither monitor breached (PSI may drift into the 'moderate' band, which the noise floor showed is normal for a clean window). Only the joined labels, weeks later, showed the drop. The response needs **new labelled data** from the new regime, or, again, a fix upstream.

</details>

## 9. Promote a bad model, then roll back *(stretch)*

> **From the deck, *Alerts People Act On, and Retraining Triggers*:** "*A retrain that ships without a **validation gate** is an automated way to deploy a worse model.*" And from the registry slide: rollback is "*repointing the alias*". This section is the deck's *An Incident, Start to Finish*, lived.

*Short on time? Skip to section 10.* This section takes about 2 minutes.

**The scenario:** a scheduled retrain runs on a fresh batch in which an upstream job **swapped the labels of two classes**, 1 and 7. Think of a renamed category, or the deck's Celsius/Fahrenheit sensor. The pipeline trains v2 without complaint. Then:
1. the **gate** compares v2 with v1 on the frozen test set, and refuses;
2. a **shadow** comparison shows how often v2 would disagree with v1;
3. we **override the gate on purpose** (this is the part real teams regret), register v2, move `champion`, and watch the running server change version and lose accuracy, with no restart;
4. we **roll back** by moving the alias back: one metadata change.

In [ ]:
require_hub(9)
SWAP = {1: 7, 7: 1}                                          # the upstream bug
y_train_bad = np.array([SWAP.get(int(label), int(label)) for label in y_train])

model_v2 = make_pipeline(**BEST_PARAMS, max_iter=MAX_EPOCHS)
with mlflow.start_run(run_name="v2-scheduled-retrain") as run:
    mlflow.log_params({**PROVENANCE, **BEST_PARAMS, "max_iter": MAX_EPOCHS,
                       "data.batch": "retrain batch - labels 1 and 7 swapped upstream (nobody knows yet)"})
    model_v2.fit(X_train, y_train_bad)
    V2_TEST_ACC, V2_PER_CLASS = evaluate(model_v2)
    mlflow.log_metrics({"test_accuracy": V2_TEST_ACC})
    V2_RUN_ID = run.info.run_id

def promotion_gate(candidate, incumbent, tolerance=0.0):
    passed = candidate >= incumbent - tolerance
    print(f"GATE: candidate {candidate:.4f} vs incumbent {incumbent:.4f} -> {'PASS' if passed else 'REFUSE'}")
    return passed

gate_passed = promotion_gate(V2_TEST_ACC, TEST_ACC)
disagree = model_v2.predict(X_test) != model_v1.predict(X_test)
print(f"SHADOW: v2 disagrees with v1 on {disagree.mean():.1%} of test images; "
      f"true classes of the disagreements: {pd.Series(y_test[disagree]).value_counts().head(3).to_dict()}")

The gate refused, and the shadow comparison says exactly where v2 goes wrong. In real life you would stop here. We override on purpose, so that you see what the registry and the prediction log buy you when someone does ship it.

In [ ]:
require_hub(9)
FORCE_PROMOTION = True          # the override. In a real pipeline, nobody should be able to set this alone.
if not gate_passed and not FORCE_PROMOTION:
    stop("The gate refused v2 and FORCE_PROMOTION is False: nothing was registered.")

BUNDLE_V2 = WORK / "registry_v2"
write_bundle(BUNDLE_V2, model_v2, V2_TEST_ACC, V2_PER_CLASS, V2_RUN_ID,
             version_note="v2, scheduled retrain. GATE REFUSED; promoted by manual override.")
commit_v2 = with_retry("Uploading v2", lambda: api.upload_folder(
    repo_id=MODEL_REPO_ID, folder_path=BUNDLE_V2, allow_patterns=SHIP_ONLY,
    commit_message=f"v2: scheduled retrain, MLflow run {V2_RUN_ID}. Gate REFUSED, promoted by override."))
MODEL_V2 = commit_v2.oid
tag_version("v2", MODEL_V2)
set_alias("challenger", MODEL_V2)            # staging: registered, not serving
tracker.set_tag(V2_RUN_ID, "registry.version", MODEL_V2)

def wait_for_version(c, sha, timeout=120):
    # Poll the live endpoint until responses come from `sha`. Returns the seconds it took.
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < timeout:
        if call(c, first_png)[0]["model_version"] == sha:
            return time.perf_counter() - t0
        time.sleep(2)
    stop(f"The server did not switch to {sha[:10]} within {timeout}s. Look at server_events()[-5:].")

def live_accuracy(c, window):
    # 200 labelled probes. `wait_for_version` watches the switch over the public URL; the probes use
    # the local address, like the rest of the bulk traffic.
    records = send(c, [test_images[i] for i in probe_pos], y_test[probe_pos], window=window)
    versions = {r["model_version"][:10] for r in records}
    return float(np.mean([r["prediction"] == y for r, y in zip(records, y_test[probe_pos])])), versions

incident = [("before", *live_accuracy(traffic_client, "probe: before"))]
print("PROMOTE:")
set_alias("champion", MODEL_V2)
switch_s = wait_for_version(client, MODEL_V2)
print(f"  the running server switched to v2 {switch_s:.0f}s after the alias moved (no restart, no redeploy)")
incident.append(("v2 in production", *live_accuracy(traffic_client, "probe: v2 live")))

**The incident.** Accuracy through the live endpoint just dropped, and every one of those responses names the model that produced it. So the first question in the deck's incident story, "*was it the model?*", is answered in seconds, from the log: yes, and it is `v2`. The previous version was never deleted, so rollback is one alias change:

In [ ]:
require_hub(9)
print("ROLL BACK:")
set_alias("champion", MODEL_V1)
rollback_s = wait_for_version(client, MODEL_V1)
print(f"  the running server is back on v1 {rollback_s:.0f}s after the alias moved")
incident.append(("after rollback", *live_accuracy(traffic_client, "probe: after rollback")))

display(pd.DataFrame(incident, columns=["moment", "live accuracy (200 labelled probes)", "model versions that answered"]))
print("the server's own record of the two alias moves:")
for e in server_events():
    if e["event"] == "alias_moved":
        print(f"  {e['ts']}  {e['old'][:10]} -> {e['new'][:10]}")
refs = api.list_repo_refs(MODEL_REPO_ID)
print("\nthe registry now:")
print("  branches:", {b.name: b.target_commit[:10] for b in refs.branches})
print("  tags    :", {t.name: t.target_commit[:10] for t in refs.tags})
print("v2 is still in the history as the audit record: archived, not served.")

## 10. Teardown

> An endpoint nobody owns is the deck's "*forgetting the owner means nobody is paged when it breaks*", and a forgotten public repo is how old models, and sometimes old data, leak. This section is not optional.

Three steps, each **verified** rather than assumed:
1. stop the server, and check that the public URL no longer answers;
2. delete the model repo, and check that the Hub answers *not found* (`RepositoryNotFoundError`);
3. print a receipt.

In [ ]:
RECEIPT = []

# 1. The server and its public URL.
stop_server()
time.sleep(3)
for label, url in (("public URL", PUBLIC_URL), ("local URL", LOCAL_URL)):
    try:
        status = requests.get(url, timeout=15).status_code
    except requests.RequestException as e:
        status = type(e).__name__
    RECEIPT.append((f"server, {label}", url, f"after stop: {status}", status != 200))

# 2. The model repo (the registry), including every version, tag and alias in it.
if api is not None:
    with_retry("Deleting the model repo", lambda: api.delete_repo(MODEL_REPO_ID, repo_type="model", missing_ok=True))
    gone = False
    for _ in range(20):
        try:
            api.repo_info(MODEL_REPO_ID, repo_type="model")
            time.sleep(3)                   # still visible: the deletion is propagating
        except RepositoryNotFoundError:
            gone = True
            break
    anonymous = requests.get(f"https://huggingface.co/api/models/{MODEL_REPO_ID}", timeout=15).status_code
    RECEIPT.append(("model repo (registry)", MODEL_REPO_ID,
                    f"API: {'RepositoryNotFoundError' if gone else 'STILL EXISTS'}; anonymous request: HTTP {anonymous}", gone))

# 3. The receipt.
print(f"TEARDOWN RECEIPT   run {RUN_ID}   {dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')}")
for what, where, evidence, ok in RECEIPT:
    print(f"  [{'x' if ok else ' '}] {what:<22} {where}\n        {evidence}")
(WORK / "teardown_receipt.json").write_text(json.dumps(RECEIPT, indent=1), encoding="utf-8")
if all(ok for *_, ok in RECEIPT):
    STATE_FILE.unlink(missing_ok=True)      # this run is finished: a re-run starts with fresh names
    print("Everything this run created is gone.")
else:
    stop("Something is still up (see the receipt). Re-run this cell, or use the sweeper below.")

Two notes on the receipt:
- **The anonymous request gets 401, not 404.** The Hub answers anonymous requests for a repo that does not exist with *401 Unauthorized*, so that it never reveals whether a *private* repo with that name exists. The authenticated `RepositoryNotFoundError` is the real proof.
- **Some things stay in this Colab VM:** the MLflow store, the bundles and the logs. None of them is public, and they vanish when the runtime is recycled. Download `mlops_lab_work/mlflow.db` first if you want to keep your runs.

### The sweeper: clean up after *any* earlier run

Perhaps an earlier run crashed before reaching this section, or you lost the Colab runtime halfway through. Its repo is still online, under a different run id.

The cell below lists every model repo in your namespace whose **full name matches the exact pattern this notebook generates**: `<namespace>/mlops-lab-<yymmdd>-<4 characters>-digits`, or with your username inside it when an instructor set `HF_ORG`. It deletes only those, and only after you set `CONFIRM_DELETE = True`.

**It cannot touch anything else:**
- it only lists **your own namespace**;
- it only looks at **model repos**, the only type this lab creates;
- it checks every name against the full pattern twice: once when listing, and again right before each delete.

A repo you created yourself would have to follow the lab's exact naming (date, four random characters, `-digits`) to match. The sweeper is idempotent: run it as often as you like.

In [ ]:
CONFIRM_DELETE = False    # set to True, then re-run this cell, to delete exactly the repos it lists

def lab_repos():
    listed = with_retry("Listing your model repos", lambda: list(api.list_models(author=NAMESPACE, search="mlops-lab-")))
    return sorted(m.id for m in listed if LAB_REPO_PATTERN.fullmatch(m.id))

require_hub(10)
targets = lab_repos()
print(f"pattern : {LAB_REPO_PATTERN.pattern}")
print(f"found   : {len(targets)} lab repo(s) in '{NAMESPACE}'" + "".join(f"\n            {t}" for t in targets))
if targets and not CONFIRM_DELETE:
    print("Nothing deleted. Set CONFIRM_DELETE = True and re-run to delete exactly the list above.")
elif targets:
    for repo_id in targets:
        if not (LAB_REPO_PATTERN.fullmatch(repo_id) and repo_id.startswith(NAMESPACE + "/")):
            stop(f"Refusing to delete {repo_id}: it does not match the lab's pattern.")      # belt and braces
        with_retry(f"Deleting {repo_id}", lambda: api.delete_repo(repo_id, repo_type="model", missing_ok=True))
        print(f"  deleted {repo_id}")
    print("remaining lab repos:", lab_repos() or "none")

> **Now revoke your token.** Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), click **Manage** next to `mlops-lab` → **Delete**. If you stored it as a Colab secret, delete that too (key icon → the bin icon).

## 11. What you did, and what to do with your own project

### The deck's "If You Do Only Eight Things"

| # | The deck says | In this lab |
|---|---|---|
| 1 | Set every seed, and log it | **Done.** `SEED` is on every run, and every `random_state` uses it. |
| 2 | Commit a lockfile; log its hash with each run | **Partly.** Versions are pinned and an environment fingerprint is logged with every run, but pins on the direct dependencies are not a lockfile of the whole transitive graph. |
| 3 | Write data to immutable dated paths; never overwrite | **Done, the Hub way.** A pinned dataset revision is an immutable snapshot, and its sha is logged with every run. |
| 4 | Start a local MLflow server and log every run, including the failures | **Done**, without even a server: a local store, and every pruned trial logged and tagged. |
| 5 | Add a schema check at training *and* serving input | **Partly.** One preprocessing contract and a `schema.json` shared by training and serving, but nothing yet *rejects* a bad input. That is Exercise 1. |
| 6 | Serialise the whole pipeline as one artifact and register it by version | **Done.** One `skops` pipeline, registered by commit, with a reviewed trust list. |
| 7 | Log every prediction with its model version | **Done**, on the client *and* on the server, joinable by `request_id`. |
| 8 | Add one alert: prediction distribution versus a fixed reference window | **Done**, with a sustained-crossing rule and a measured noise floor. |

Six done, two partly: the six the lab promised.

### Audit your capstone: the ML Test Score

> **From the deck, *A Rubric to Audit Yourself*:** Breck et al. list 28 tests in four sections. Score each **0** (not done), **0.5** (done by hand, with the result written down and shared) or **1** (run automatically and repeatedly). "***The final score is the minimum of the four section scores.*** *Beautiful model development with zero monitoring scores zero.*"

Fill in the dictionary below for **your own capstone project**, not for this lab, and re-run the cell. The test names are the paper's own: Breck, Cai, Nielsen, Salib & Sculley, [*The ML Test Score*, IEEE Big Data 2017](https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/).

In [ ]:
# Score YOUR capstone: 0 = not done, 0.5 = done by hand and written down, 1 = automated and repeated.
ML_TEST_SCORE = {
    "Features and data": {
        "Feature expectations are captured in a schema": 0,
        "All features are beneficial": 0,
        "No feature's cost is too much": 0,
        "Features adhere to meta-level requirements": 0,
        "The data pipeline has appropriate privacy controls": 0,
        "New features can be added quickly": 0,
        "All input feature code is tested": 0,
    },
    "Model development": {
        "Model specs are reviewed and submitted": 0,
        "Offline and online metrics correlate": 0,
        "All hyperparameters have been tuned": 0,
        "The impact of model staleness is known": 0,
        "A simpler model is not better": 0,
        "Model quality is sufficient on important data slices": 0,
        "The model is tested for considerations of inclusion": 0,
    },
    "ML infrastructure": {
        "Training is reproducible": 0,
        "Model specs are unit tested": 0,
        "The ML pipeline is integration tested": 0,
        "Model quality is validated before serving": 0,
        "The model is debuggable": 0,
        "Models are canaried before serving": 0,
        "Serving models can be rolled back": 0,
    },
    "Monitoring": {
        "Dependency changes result in notification": 0,
        "Data invariants hold for inputs": 0,
        "Training and serving are not skewed": 0,
        "Models are not too stale": 0,
        "Models are numerically stable": 0,
        "Computing performance has not regressed": 0,
        "Prediction quality has not regressed": 0,
    },
}
section_scores = {section: sum(tests.values()) for section, tests in ML_TEST_SCORE.items()}
for section, score in section_scores.items():
    print(f"  {section:<20} {score:.1f} / 7")
print(f"ML Test Score = minimum over the four sections = {min(section_scores.values()):.1f}")
print(f"Invest next in: {min(section_scores, key=section_scores.get)}")

### Exercises

1. **A health check and a rejection path.** A blank canvas currently gets a confident-looking guess; try it on your phone. Add a `healthz` endpoint: a Gradio function with `api_name="healthz"` that returns the status and the model version. Then make `predict` **reject** inputs that break the contract, with a `gr.Error`: no ink, far too much ink, an image over 4000 px. This is the deck's "*validate the input schema*", and item 5 of the eight things. Where should the limits live so that training and serving agree?
2. **Batch instead of online.** Write `score_folder(folder) -> DataFrame`. It should resolve `champion` once, load that version, and score a directory of images into a table. Then argue the cost difference with the deck's *Batch, Online, Streaming* table: what does an always-on process cost per day at 50 requests a day, and what does one nightly job cost?
3. **A canary.** Change `app.py` so that it loads **both** `champion` and `challenger`, and serves `challenger` to a configurable percentage of requests. Hash the `request_id` so the split is reproducible, and log which version answered. How many labelled requests do you need before you can tell a 2-point accuracy drop from noise? (The deck's *Never Flip the Switch* slide.)
4. **Register on push.** Sketch a GitHub Actions workflow that runs on every push to `main`. It should retrain on the pinned snapshot, run the reload test and the gate, and only if both pass, upload the bundle and move `challenger`. Where does the token live? (Hint: with the Hub's [Trusted Publishers](https://huggingface.co/docs/hub/trusted-publishers), a CI job does not need to hold a long-lived token at all.)

# Appendix: Plan B, no Hugging Face account

Use this only if you cannot create a Hugging Face account. At the token prompt in Setup, press Enter; `PLAN_B` becomes `True`. Then the same `app.py` serves the **local** bundle folder from section 4 (`MODEL_DIR`) instead of the registry, still through `demo.launch(share=True)`, so you still get a temporary public URL. Sections 7 and 8 run unchanged against it.

**Run:** Setup, sections 1-4, section 6, sections 7-8, then the first cell of section 10 (it stops the server). **Skip:** section 5 and section 9 (they need a registry), and the sweeper.

Be clear about what this is: **a tunnel to a process in your notebook, not a deployment.**
- There is **no registry**: no version history, and no alias to promote or roll back.
- The "model version" in each response is a hash of the local file.
- There is nothing on the Hub to delete.
- When the kernel dies, the URL dies with it.

Everything the registry sections teach, lineage, promotion, rollback, and a teardown you can verify, is exactly what Plan B lacks. That makes it a useful comparison in its own right.